## 0. Task

Summarization, BERTScore. **Selected:** encoder–decoder extractive, 720-word budget.


In [ ]:
import os, sys, types, random, json, math, zipfile
from pathlib import Path

_BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}

def _register(name):
    global _BEFORE
    mod = types.ModuleType(name)
    mod.__name__ = name
    mod.__file__ = name + ".py"
    for key, value in list(globals().items()):
        if key not in _BEFORE:
            setattr(mod, key, value)
    sys.modules[name] = mod
    _BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}
    print("in this notebook:", name)

print("This notebook is self-contained (starter style). RUN_TRAINING = False for a viva.")


## 1. Check GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


## 2. Phase-1 body


In [ ]:
"""Byte-level BPE for CS781 Capstone Phase 1.

This is trained from scratch on the capstone corpus. It does not use
SentencePiece, Hugging Face tokenizers, or the GPT-2 vocabulary.

Assignment 1 merges the pair that occurs most often. On legal text that
fills the vocabulary with pieces of very common words such as "the" and
"of", even when those letters usually belong to other words.

Here a pair must occur at least `min_pair_count` times. Among those pairs
the one with the highest Dice-weighted count is merged:

    score(x, y) = C(xy) * 2 * C(xy) / (C(x) + C(y))

The fraction is near 1 when x and y almost always appear as this pair, and
near 0 when they usually appear apart. Multiplying by C(xy) keeps frequent
pairs ahead of rare ones. If two pairs have the same score, the higher count
wins. If the counts are also equal, the lexicographically smaller pair wins,
which is the Assignment 1 tie-break.

Merges stay inside a whitespace-separated word. A word-end marker is part of
that word, so a frequent whole word can become one token. Spaces stay
spaces, and uppercase stays uppercase. The first four vocabulary ids are
<pad>, <unk>, <s>, </s>. Every byte 0-255 has an id, so ordinary text does
not need <unk>.
"""

from __future__ import annotations

import heapq
import json
import re
from collections import Counter


END = 256  # word-end marker, not a UTF-8 byte
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
SPECIALS = ("<pad>", "<unk>", "<s>", "</s>")
# Model ids 0-3 are the specials above. Internal symbol 0 (byte 0) is model id 4.
ID_OFFSET = 4
BASE_SYMBOLS = 257  # bytes 0-255 plus END
TOKEN_RE = re.compile(r"\s+|\S+")


def iter_texts(records):
    """Yield strings from a list, a dict of records, or a single record."""
    if isinstance(records, list):
        rows = records
    elif isinstance(records, dict):
        if isinstance(records.get("text"), str):
            rows = [records]
        else:
            rows = list(records.values())
    else:
        raise TypeError("records must be a list or a dict")
    for row in rows:
        if isinstance(row, str):
            yield row
        elif isinstance(row, dict) and isinstance(row.get("text"), str):
            yield row["text"]


class LegalBPE:
    def __init__(self):
        self.score_name = "dice"
        self.min_pair_count = 2
        self.requested_vocab_size = ID_OFFSET + BASE_SYMBOLS
        self.merges = []  # list of (left_internal, right_internal), in learn order
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.merge_rank = {}
        self.merge_to = {}
        self.keys = {}
        self.history = []
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------------
    # Public attributes the Phase 1 notebook uses
    # ------------------------------------------------------------------

    @property
    def pad_token_id(self):
        return PAD_ID

    @property
    def unk_token_id(self):
        return UNK_ID

    @property
    def bos_token_id(self):
        return BOS_ID

    @property
    def eos_token_id(self):
        return EOS_ID

    pad_token = SPECIALS[0]
    unk_token = SPECIALS[1]
    bos_token = SPECIALS[2]
    eos_token = SPECIALS[3]

    def __len__(self):
        return ID_OFFSET + BASE_SYMBOLS + len(self.merges)

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    @classmethod
    def train(cls, texts, vocab_size=8192, min_pair_count=2, score="dice", max_chars=None):
        if score not in ("dice", "frequency"):
            raise ValueError("score must be 'dice' or 'frequency'")
        minimum = ID_OFFSET + BASE_SYMBOLS + 1
        if vocab_size < minimum:
            raise ValueError(f"vocab_size must be at least {minimum}")

        model = cls()
        model.score_name = score
        model.min_pair_count = int(min_pair_count)
        model.requested_vocab_size = int(vocab_size)
        target_merges = int(vocab_size) - (ID_OFFSET + BASE_SYMBOLS)

        word_freq = Counter()
        documents = 0
        seen_chars = 0
        for text in texts:
            documents += 1
            if not isinstance(text, str) or text == "":
                continue
            if max_chars is not None and seen_chars >= max_chars:
                break
            seen_chars += len(text)
            for tok in TOKEN_RE.findall(text):
                word_freq[model._token_symbols(tok)] += 1
            if documents % 100000 == 0:
                print(f"BPE counted {documents:,} documents, {len(word_freq):,} unique words")

        if not word_freq:
            raise ValueError("BPE training saw no text")

        print(
            f"BPE documents: {documents:,} | characters: {seen_chars:,} | "
            f"unique words: {len(word_freq):,} | score: {score} | target vocab: {vocab_size}"
        )
        model._learn(word_freq, target_merges)
        print(f"BPE finished. Vocabulary size: {len(model):,}. Merges: {len(model.merges):,}.")
        for left, right, count, value in model.history[:12]:
            print(f"  merge {left!r} + {right!r}  count={count}  score={value:.2f}")
        return model

    def _learn(self, word_freq, target_merges):
        self.word_freq = dict(word_freq)
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.version = {}
        self.heap = []
        self.history = []
        self.merges = []
        self.merge_rank = {}
        self.merge_to = {}
        self._cache = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.keys = {i: self._key_for(i) for i in range(257)}

        dirty_pairs = set()
        dirty_syms = set()
        for word, freq in self.word_freq.items():
            self._register(word, freq, dirty_pairs, dirty_syms)
        self._rebuild_heap()

        next_id = 257
        while len(self.merges) < target_merges:
            if len(self.heap) > max(1000, 8 * len(self.pair_counts)):
                self._rebuild_heap()
            chosen = self._pop_best()
            if chosen is None:
                print(
                    f"BPE stopped early at vocab {len(self)}. "
                    f"No pair occurs at least {self.min_pair_count} times."
                )
                break
            left, right = chosen
            count = self.pair_counts[chosen]
            value = self._score(chosen)
            new_id = next_id
            next_id += 1
            self.symbol_bytes[new_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            self.keys[new_id] = self._key_for(new_id)
            self.merges.append((left, right))
            self.history.append((self._pretty(left), self._pretty(right), int(count), float(value)))

            dirty_pairs = set()
            dirty_syms = set()
            for word in list(self.pair_words.get(chosen, ())):
                freq = self.word_freq.pop(word, None)
                if freq is None:
                    continue
                self._unregister(word, freq, dirty_pairs, dirty_syms)
                merged = _merge_symbols(word, left, right, new_id)
                if merged in self.word_freq:
                    self.word_freq[merged] += freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)
                else:
                    self.word_freq[merged] = freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)

            affected = set(dirty_pairs)
            for sym in dirty_syms:
                affected.update(self.sym_pairs.get(sym, ()))
            affected.add(chosen)
            for pair in affected:
                self._push(pair)

            done = len(self.merges)
            if done <= 10 or done % 500 == 0:
                print(
                    f"  [{done}/{target_merges}] {self.history[-1][0]!r} + "
                    f"{self.history[-1][1]!r}  count={count}  score={value:.2f}"
                )

        self._rebuild_tables()
        # Training indexes are large. Drop them before save.
        self.word_freq = {}
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.heap = []

    def _register(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] += freq
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] += freq * times
            self.pair_words.setdefault(pair, set()).add(word)
            for sym in pair:
                self.sym_pairs.setdefault(sym, set()).add(pair)
            dirty_pairs.add(pair)

    def _unregister(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] -= freq
            if self.sym_counts[sym] <= 0:
                del self.sym_counts[sym]
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] -= freq * times
            words = self.pair_words.get(pair)
            if words is not None:
                words.discard(word)
                if not words:
                    del self.pair_words[pair]
            if self.pair_counts[pair] <= 0:
                del self.pair_counts[pair]
                for sym in pair:
                    bucket = self.sym_pairs.get(sym)
                    if bucket is not None:
                        bucket.discard(pair)
                        if not bucket:
                            del self.sym_pairs[sym]
                self.version[pair] = self.version.get(pair, 0) + 1
            else:
                dirty_pairs.add(pair)

    def _score(self, pair):
        count = self.pair_counts.get(pair, 0)
        if count < self.min_pair_count:
            return None
        if self.score_name == "frequency":
            return float(count)
        left = self.sym_counts.get(pair[0], 0)
        right = self.sym_counts.get(pair[1], 0)
        if left <= 0 or right <= 0:
            return None
        return (2.0 * count * count) / (left + right)

    def _push(self, pair):
        value = self._score(pair)
        if value is None:
            return
        version = self.version.get(pair, 0) + 1
        self.version[pair] = version
        count = self.pair_counts[pair]
        heapq.heappush(
            self.heap,
            (
                -value,
                -count,
                self.keys[pair[0]],
                self.keys[pair[1]],
                pair[0],
                pair[1],
                version,
            ),
        )

    def _rebuild_heap(self):
        self.heap = []
        for pair in list(self.pair_counts):
            self._push(pair)

    def _pop_best(self):
        while self.heap:
            _neg_value, _neg_count, _left_key, _right_key, left, right, version = heapq.heappop(self.heap)
            pair = (left, right)
            if self.version.get(pair) != version:
                continue
            if self._score(pair) is None:
                continue
            return pair
        return None

    def _rebuild_tables(self):
        self.merge_rank = {}
        self.merge_to = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        next_id = 257
        for rank, (left, right) in enumerate(self.merges):
            self.merge_rank[(left, right)] = rank
            self.merge_to[(left, right)] = next_id
            self.symbol_bytes[next_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            next_id += 1
        self.keys = {i: self._key_for(i) for i in self.symbol_bytes}
        self._cache = {}

    def _key_for(self, sym):
        parts = []
        for piece in self.symbol_bytes[sym]:
            parts.append("</w>" if piece == END else f"{piece:02x}")
        return "".join(parts)

    def _pretty(self, sym):
        raw = []
        end = False
        for piece in self.symbol_bytes[sym]:
            if piece == END:
                end = True
            else:
                raw.append(piece)
        try:
            text = bytes(raw).decode("utf-8")
            shown = text.encode("ascii", errors="backslashreplace").decode("ascii")
        except UnicodeDecodeError:
            shown = "".join(f"<{byte:02x}>" for byte in raw)
        return shown + "</w>" if end else shown

    def _token_symbols(self, token):
        raw = tuple(token.encode("utf-8"))
        if token.isspace():
            return raw
        return raw + (END,)

    # ------------------------------------------------------------------
    # Encode / decode
    # ------------------------------------------------------------------

    def encode(self, text, add_special_tokens=False):
        ids = []
        if add_special_tokens:
            ids.append(BOS_ID)
        if text:
            for tok in TOKEN_RE.findall(text):
                ids.extend(self._encode_token(tok))
        if add_special_tokens:
            ids.append(EOS_ID)
        return ids

    def encode_with_offsets(self, text):
        """Map text to token ids and the exact character span of each token.

        Merges never cross a whitespace boundary, and the word-end marker
        does not consume a character. A span is a half-open [start, end)
        into `text`, which is the same convention as the Phase 2 gold files.
        """
        ids = []
        spans = []
        for match in TOKEN_RE.finditer(text):
            token = match.group()
            origin = match.start()
            char_of_byte = []
            for index, char in enumerate(token):
                char_of_byte.extend([origin + index] * len(char.encode("utf-8")))
            byte_at = 0
            for model_id in self._encode_token(token):
                pieces = self.symbol_bytes.get(int(model_id) - ID_OFFSET, ())
                nbytes = sum(1 for piece in pieces if piece != END)
                if nbytes <= 0 or byte_at >= len(char_of_byte):
                    continue
                stop = min(byte_at + nbytes, len(char_of_byte))
                left = char_of_byte[byte_at]
                right = char_of_byte[stop - 1] + 1
                byte_at = stop
                if right > left:
                    ids.append(int(model_id))
                    spans.append((left, right))
        return ids, spans

    def _encode_token(self, token):
        cached = self._cache.get(token)
        if cached is None:
            symbols = list(self._token_symbols(token))
            while len(symbols) >= 2:
                best_rank = None
                best_index = None
                for i in range(len(symbols) - 1):
                    rank = self.merge_rank.get((symbols[i], symbols[i + 1]))
                    if rank is not None and (best_rank is None or rank < best_rank):
                        best_rank = rank
                        best_index = i
                if best_index is None:
                    break
                pair = (symbols[best_index], symbols[best_index + 1])
                symbols = symbols[:best_index] + [self.merge_to[pair]] + symbols[best_index + 2 :]
            cached = tuple(sym + ID_OFFSET for sym in symbols)
            self._cache[token] = cached
        return cached

    def decode(self, ids, skip_special_tokens=True):
        raw = []
        for model_id in ids:
            model_id = int(model_id)
            if model_id < ID_OFFSET:
                continue
            internal = model_id - ID_OFFSET
            pieces = self.symbol_bytes.get(internal)
            if pieces is None:
                continue
            for piece in pieces:
                if piece != END:
                    raw.append(piece)
        return bytes(raw).decode("utf-8", errors="replace")

    def batch_encode(self, texts, max_length=None, padding="left"):
        """Pad a batch for generation. Keep the end of a long prompt."""
        rows = []
        for text in texts:
            ids = self.encode(text, add_special_tokens=False)
            if max_length is not None and len(ids) > max_length:
                ids = ids[-max_length:]
            rows.append(ids)
        width = max((len(row) for row in rows), default=0)
        input_ids = []
        attention_mask = []
        for row in rows:
            pad = width - len(row)
            if padding == "left":
                input_ids.append([PAD_ID] * pad + row)
                attention_mask.append([0] * pad + [1] * len(row))
            else:
                input_ids.append(row + [PAD_ID] * pad)
                attention_mask.append([1] * len(row) + [0] * pad)
        return input_ids, attention_mask

    # ------------------------------------------------------------------
    # Save / load
    # ------------------------------------------------------------------

    def save(self, path):
        payload = {
            "format": "cs781-legal-bpe-v1",
            "score": self.score_name,
            "min_pair_count": self.min_pair_count,
            "vocab_size": len(self),
            "specials": list(SPECIALS),
            "merges": self.merges,
            "preview": [f"{left} + {right}" for left, right, _count, _value in self.history[:40]],
        }
        with open(path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False)
        print("Saved BPE to", path)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if payload.get("format") != "cs781-legal-bpe-v1":
            raise ValueError(f"{path} is not a cs781-legal-bpe-v1 file")
        model = cls()
        model.score_name = payload.get("score", "dice")
        model.min_pair_count = int(payload.get("min_pair_count", 2))
        model.merges = [(int(left), int(right)) for left, right in payload["merges"]]
        model.requested_vocab_size = int(payload.get("vocab_size", len(model)))
        model.history = []
        model._rebuild_tables()
        return model


def _merge_symbols(word, left, right, new_id):
    merged = []
    index = 0
    size = len(word)
    while index < size:
        if index + 1 < size and word[index] == left and word[index + 1] == right:
            merged.append(new_id)
            index += 2
        else:
            merged.append(word[index])
            index += 1
    return tuple(merged)


def _check():
    sticky = ["ab"] * 6
    loose = ["mn", "mp", "mq", "mr", "ms"] * 10
    one_merge = ID_OFFSET + BASE_SYMBOLS + 1

    dice = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="dice")
    freq = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="frequency")
    assert dice.history[0][0] == "a" and dice.history[0][1] == "b", dice.history[0]
    assert freq.history[0][0] in list("mnmpqrms") or freq.history[0][0].startswith("m"), freq.history[0]
    assert not (freq.history[0][0] == "a" and freq.history[0][1] == "b"), freq.history[0]

    samples = [
        "The Supreme Court held that the petitioner is entitled to relief.",
        "Section 302 of the Indian Penal Code\nwas applied.",
        "court Court COURT",
        "न्यायालय ने आदेश दिया।",
        "  spaced   words\tand tabs",
    ]
    full = LegalBPE.train(samples * 30, vocab_size=400, min_pair_count=2, score="dice")
    for text in samples:
        assert full.decode(full.encode(text)) == text, text
    assert full.encode("Court") != full.encode("court")
    joined = full.decode(full.encode("ab cd"))
    assert joined == "ab cd"
    # A space is its own token, so a merge cannot glue the b in "ab" to the c in "cd".
    encoded = full.encode("ab cd")
    assert len(encoded) >= 2

    path = "_legal_bpe_selfcheck.json"
    full.save(path)
    loaded = LegalBPE.load(path)
    for text in samples:
        assert loaded.decode(loaded.encode(text)) == text
    assert len(loaded) == len(full)
    ids, mask = loaded.batch_encode(["short", "a longer prompt"], max_length=8, padding="left")
    assert len(ids[0]) == len(ids[1])
    assert mask[0][0] == 0 or len(loaded.encode("short")) == len(ids[0])
    import os
    os.remove(path)
    print("legal_bpe self-check passed")


In [ ]:
_register('legal_bpe')


In [ ]:
"""Four Phase-1 language models, all trained from scratch.

They share a token embedding with the output layer. None of them loads a
pretrained checkpoint, and none of them uses nn.Transformer or
nn.MultiheadAttention.

The four designs are different in structure, not just in width or learning rate:

decoder
    Causal Transformer. A token may look only at earlier tokens. Position is a
    learned vector. The feed-forward layer is a GELU MLP. Pre-norm LayerNorm.

encdec
    The first half of a training block is read by a bidirectional encoder.
    A causal decoder writes the second half and cross-attends to the encoder.
    At test time the prompt is encoded, then the continuation is decoded.

hybrid
    Same size as the decoder, but most layers see only a local window of
    earlier tokens. Every fourth layer is full causal attention, so a long
    legal reference can still move through the network.

efficient
    RoPE instead of learned positions, RMSNorm instead of LayerNorm, SwiGLU
    instead of GELU, and grouped-query attention: 16 query heads share 4
    key/value heads.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


VOCAB_SIZE = 8192
BLOCK = 1024


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


class RMSNorm(nn.Module):
    def __init__(self, n_embd, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embd))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


def _init_linear(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)


def apply_rope(x):
    # x: batch, heads, time, dim. Pairs (0,1), (2,3), ... are rotated by position.
    _batch, _heads, steps, dim = x.shape
    half = dim // 2
    freq = 1.0 / (10000 ** (torch.arange(half, device=x.device, dtype=torch.float32) / half))
    pos = torch.arange(steps, device=x.device, dtype=torch.float32)
    angles = torch.outer(pos, freq)
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    even = x[..., 0::2].float()
    odd = x[..., 1::2].float()
    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos
    stacked = torch.stack((rotated_even, rotated_odd), dim=-1)
    return stacked.flatten(-2).to(dtype=x.dtype)


class CausalAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must divide into equal heads")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.window = window
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            banned = self.causal[:steps, :steps]
            if self.window is not None:
                positions = torch.arange(steps, device=x.device)
                too_far = positions[None, :] < (positions[:, None] - (self.window - 1))
                banned = banned | too_far
            scores = scores.masked_fill(banned, float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size):
        super().__init__()
        if n_embd % n_head != 0 or n_head % n_kv_head != 0:
            raise ValueError("heads must divide the width and the kv heads")
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, _channels = x.shape
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        value = self.v(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        query = apply_rope(query)
        key = apply_rope(key)
        groups = self.n_head // self.n_kv_head
        key = key.repeat_interleave(groups, dim=1)
        value = value.repeat_interleave(groups, dim=1)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            scores = scores.masked_fill(self.causal[:steps, :steps], float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, -1)
        return self.proj(mixed)


class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_embd, bias=False)
        self.v = nn.Linear(n_embd, n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, memory, memory_mask=None):
        batch, steps, channels = x.shape
        mem_steps = memory.size(1)
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        value = self.v(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if memory_mask is not None:
            scores = scores.masked_fill(~memory_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class BidirectionalAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GeluMLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.up = nn.Linear(n_embd, 4 * n_embd)
        self.down = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class SwiGLU(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden = int(8 * n_embd / 3)
        self.gate = nn.Linear(n_embd, hidden, bias=False)
        self.up = nn.Linear(n_embd, hidden, bias=False)
        self.down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = CausalAttention(n_embd, n_head, block_size, window=window)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EfficientBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, block_size)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = BidirectionalAttention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class DecoderCrossBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.self_attn = CausalAttention(n_embd, n_head, block_size)
        self.norm2 = nn.LayerNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head)
        self.norm3 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, pad_mask=None, memory_mask=None):
        x = x + self.drop(self.self_attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, memory_mask=memory_mask))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x


class TiedLM(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, use_positions=True):
        super().__init__()
        self.block_size = block_size
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd) if use_positions else None
        self.drop = nn.Dropout(0.1)
        self.norm = None

    def token_plus_position(self, idx):
        x = self.embed(idx)
        if self.pos is not None:
            steps = idx.size(1)
            positions = torch.arange(steps, device=idx.device)
            x = x + self.pos(positions)
        return self.drop(x)

    def logits_from(self, x):
        x = self.norm(x)
        return F.linear(x, self.embed.weight)

    def run_blocks(self, blocks, x, pad_mask=None):
        for block in blocks:
            if self.training:
                x = checkpoint(block, x, pad_mask, use_reentrant=False)
            else:
                x = block(x, pad_mask=pad_mask)
        return x


class DecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=8, n_head=16, block_size=BLOCK, windowed=False):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        blocks = []
        for layer in range(n_layer):
            window = None
            if windowed and layer % 4 != 3:
                window = 256
            blocks.append(DecoderBlock(n_embd, n_head, block_size, window=window))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


class EncoderDecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=768, n_layer=6, n_head=12, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        self.encoder = nn.ModuleList(EncoderBlock(n_embd, n_head) for _ in range(n_layer))
        self.decoder = nn.ModuleList(DecoderCrossBlock(n_embd, n_head, block_size) for _ in range(n_layer))
        self.norm = nn.LayerNorm(n_embd)
        self.enc_norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def encode(self, idx, pad_mask=None):
        x = self.token_plus_position(idx)
        x = self.run_blocks(self.encoder, x, pad_mask)
        return self.enc_norm(x)

    def decode(self, idx, memory, pad_mask=None, memory_mask=None):
        x = self.token_plus_position(idx)
        for block in self.decoder:
            if self.training:
                x = checkpoint(block, x, memory, pad_mask, memory_mask, use_reentrant=False)
            else:
                x = block(x, memory, pad_mask=pad_mask, memory_mask=memory_mask)
        return self.logits_from(x)

    def forward(self, idx, labels=None, pad_mask=None):
        steps = idx.size(1)
        if steps < 2 or steps > self.block_size:
            raise ValueError("encoder-decoder training needs a block of length 2..context")
        split = steps // 2
        memory = self.encode(idx[:, :split], None if pad_mask is None else pad_mask[:, :split])
        dec_in = idx[:, split - 1 : -1]
        target = idx[:, split:]
        logits = self.decode(dec_in, memory)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
        return logits, loss


class EfficientLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=10, n_head=16, n_kv_head=4, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=False)
        self.blocks = nn.ModuleList(
            EfficientBlock(n_embd, n_head, n_kv_head, block_size) for _ in range(n_layer)
        )
        self.norm = RMSNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


def build_model(name, vocab_size=VOCAB_SIZE):
    if name == "decoder":
        model = DecoderLM(vocab_size=vocab_size)
    elif name == "encdec":
        model = EncoderDecoderLM(vocab_size=vocab_size)
    elif name == "hybrid":
        model = DecoderLM(vocab_size=vocab_size, windowed=True)
    elif name == "efficient":
        model = EfficientLM(vocab_size=vocab_size)
    else:
        raise ValueError("architecture must be decoder, encdec, hybrid, or efficient")
    total = count_parameters(model)
    if not 50_000_000 <= total <= 300_000_000:
        raise RuntimeError(f"{name} has {total} parameters, outside 50M-300M")
    return model


def sample_top_p(logits, top_p):
    sorted_logits, sorted_index = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(probs, dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    choice = torch.multinomial(torch.softmax(sorted_logits, dim=-1), num_samples=1)
    return torch.gather(sorted_index, -1, choice).squeeze(-1)


def _apply_stop(stop_fn, produced, finished, step, check_every):
    if stop_fn is None or (step + 1) % check_every != 0:
        return finished
    for position in torch.where(~finished)[0].tolist():
        if stop_fn(produced[position]):
            finished[position] = True
    return finished


@torch.no_grad()
def generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32, min_new_tokens=0):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    produced = [[] for _ in range(idx.size(0))]
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1, max=model.block_size)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if bool((active & (lengths >= model.block_size)).any()):
            idx = idx[:, 1:]
            pad_mask = pad_mask[:, 1:]
            lengths = (lengths - 1).clamp(min=1)
        if bool((lengths >= idx.size(1)).any()):
            idx = torch.cat([idx, idx.new_full((idx.size(0), 1), pad_id)], dim=1)
            pad_mask = torch.cat(
                [pad_mask, torch.zeros(idx.size(0), 1, dtype=torch.bool, device=idx.device)],
                dim=1,
            )
        limit = int(lengths.max().item())
        logits, _ = model(idx[:, :limit], pad_mask=pad_mask[:, :limit])
        last = (lengths - 1).clamp(min=0)
        row = torch.arange(idx.size(0), device=idx.device)
        next_logits = logits[row, last].float()
        if min_new_tokens and eos_id is not None:
            for position in range(idx.size(0)):
                if len(produced[position]) < min_new_tokens:
                    next_logits[position, eos_id] = -1e9
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        idx[row[active], lengths[active]] = next_id[active]
        pad_mask[row[active], lengths[active]] = True
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        lengths = lengths + active.long()
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


@torch.no_grad()
def generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1)
    memory = model.encode(idx, pad_mask)
    row = torch.arange(idx.size(0), device=idx.device)
    dec = idx[row, lengths - 1].unsqueeze(1)
    produced = [[] for _ in range(idx.size(0))]
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if dec.size(1) >= model.block_size:
            dec = dec[:, 1:]
        logits = model.decode(dec, memory, memory_mask=pad_mask)
        next_logits = logits[:, -1, :]
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        next_id = torch.where(active, next_id, torch.full_like(next_id, eos_id or pad_id))
        dec = torch.cat([dec, next_id.unsqueeze(1)], dim=1)
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


def generate(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, min_new_tokens=0):
    if isinstance(model, EncoderDecoderLM):
        return generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn)
    return generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn,
                           min_new_tokens=min_new_tokens)


def _check():
    torch.manual_seed(0)
    for name in ("decoder", "encdec", "hybrid", "efficient"):
        model = build_model(name, vocab_size=VOCAB_SIZE)
        total = count_parameters(model)
        print(name, f"{total / 1e6:.2f}M")
        model.train()
        idx = torch.randint(1, 100, (2, 32))
        _logits, loss = model(idx, labels=idx)
        loss.backward()
        model.eval()
        produced = generate(model, idx[:, :8], max_new_tokens=3, temperature=0.0, top_p=1.0, eos_id=None, pad_id=0)
        assert len(produced) == 2
        assert all(len(row) == 3 for row in produced)
    print("phase1 model check passed")


In [ ]:
_register('phase1_models')


In [ ]:
"""Phase 2 legal named-entity recognition, starting from the Phase 1 models.

The score is strict macro-F1. A span counts only when the start character,
the end character, and the label all match. The previous draft of this file
trained a word BiLSTM and stopped at 400 words, so entities later in a
judgment were never predicted. This version:

  * loads one of the four Phase 1 models and adds a token-classification head
  * aligns the Phase 1 byte-level BPE to character offsets
  * slides a window across the whole document, including every test id
  * trains with full fine-tuning, LoRA, LoRMA+, or knowledge distillation

LoRMA+ is the multiplicative adapter from Bihany, Patel, and Modi (2025):

    h = (I + (alpha / r) B A) W x

with B initialised at zero, so the first step matches the Phase 1 model.

The four required submissions, each from a different Phase 1 architecture:

    python phase2_lner.py --arch efficient --method lorma
    python phase2_lner.py --arch decoder --method full
    python phase2_lner.py --arch hybrid --method lora
    python phase2_lner.py --arch encdec --method distill --teacher <efficient checkpoint>

Official deadline: 11 October 2026, 11:59 PM.
"""

import argparse
import json
import os
import random
import shutil
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F

from legal_bpe import LegalBPE
from phase1_models import EncoderDecoderLM, build_model


SEED = 42
LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]
LOCAL = r"C:\Users\Anshuman Raj\CS781"
DEFAULT_WEIGHTS = {
    "efficient": os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\model.pt"),
    "decoder": os.path.join(LOCAL, r"local_weights\cs781_phase1_decoder\model.pt"),
    "hybrid": os.path.join(LOCAL, r"local_weights\cs781_phase1_hybrid\model.pt"),
    "encdec": os.path.join(LOCAL, r"encdec_run2\cs781_phase1_encdec\model.pt"),
}
DEFAULT_BPE = os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\legal_bpe.json")
REPO = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = None
PROPAGATE = False
_GENERIC = {
    "state",
    "court",
    "india",
    "union",
    "government",
    "appellant",
    "appellants",
    "respondent",
    "respondents",
    "petitioner",
    "petitioners",
    "others",
    "other",
    "ors",
    "anr",
    "the",
    "shri",
    "smt",
    "and",
}


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fetch(name):
    if DATA_DIR:
        local = os.path.join(DATA_DIR, *name.split("/"))
        if os.path.isfile(local):
            print("local", name, flush=True)
            return local
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(repo_id=REPO, filename=name, repo_type="dataset")
    print("got", name, flush=True)
    return path


def read_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def load_fold(text_name, gold_name=None):
    texts = read_json(fetch(text_name))
    golds = read_json(fetch(gold_name)) if gold_name else {}
    rows = []
    for doc_id, text in texts.items():
        spans = []
        for item in golds.get(doc_id, []):
            start, end, label = int(item[0]), int(item[1]), str(item[2])
            if label in LABELS and 0 <= start < end <= len(text):
                spans.append((start, end, label))
        rows.append((str(doc_id), text, spans))
    return rows


def tag_names_of():
    names = ["O"]
    for label in LABELS:
        names.append("B-" + label)
        names.append("I-" + label)
    return names


def bio_tags(token_spans, entities):
    """Label each token with the entity that covers the most of its characters."""
    assigned = [-1] * len(token_spans)
    for index, (left, right) in enumerate(token_spans):
        best = -1
        best_overlap = 0
        for entity_index, (start, end, _label) in enumerate(entities):
            overlap = min(right, end) - max(left, start)
            if overlap > best_overlap:
                best_overlap = overlap
                best = entity_index
        if best_overlap > 0:
            assigned[index] = best
    tags = []
    for index, entity_index in enumerate(assigned):
        if entity_index < 0:
            tags.append("O")
            continue
        label = entities[entity_index][2]
        previous = assigned[index - 1] if index else -1
        prefix = "I-" if previous == entity_index else "B-"
        tags.append(prefix + label)
    return tags


def make_windows(ids, spans, tags, width, stride):
    rows = []
    start = 0
    while start < len(ids):
        stop = min(start + width, len(ids))
        rows.append(
            {
                "ids": ids[start:stop],
                "spans": spans[start:stop],
                "tags": tags[start:stop],
                "offset": start,
            }
        )
        if stop == len(ids):
            break
        start += stride
    return rows


def prepare_document(tokenizer, doc_id, text, entities, width, stride, tag_index):
    ids, spans = tokenizer.encode_with_offsets(text)
    tags = bio_tags(spans, entities)
    windows = make_windows(ids, spans, tags, width, stride)
    for window in windows:
        window["doc_id"] = doc_id
        window["target"] = [tag_index[tag] for tag in window["tags"]]
    return ids, spans, tags, windows


def boundary_report(tokenizer, rows):
    """How often a gold span lands exactly on token boundaries."""
    covered = 0
    total = 0
    tiled = 0
    shown = 0
    for _doc_id, text, entities in rows:
        ids, spans = tokenizer.encode_with_offsets(text)
        if spans and spans[0][0] == 0 and spans[-1][1] == len(text):
            if all(right == next_left for (_left, right), (next_left, _next_right) in zip(spans, spans[1:])):
                tiled += 1
        starts = {left for left, _right in spans}
        ends = {right for _left, right in spans}
        for start, end, label in entities:
            total += 1
            if start in starts and end in ends:
                covered += 1
            elif shown < 5:
                shown += 1
                print(
                    "unaligned",
                    label,
                    repr(text[start:end][:60]),
                    "around",
                    repr(text[max(0, start - 8) : end + 8][:80]),
                )
    print(
        f"documents tiled by tokens {tiled}/{len(rows)}  "
        f"gold spans on token boundaries {covered}/{total}",
        flush=True,
    )
    return covered, total


class LoRALinear(nn.Module):
    """Additive low-rank update. The base Phase 1 matrix stays frozen."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        out_features, in_features = base.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        update = (x @ self.A.transpose(0, 1)) @ self.B.transpose(0, 1)
        return self.base(x) + self.scale * update


class LoRMALinear(nn.Module):
    """LoRMA+ : h = (I + (alpha / r) B A) W x, with B starting at zero."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        out_features, _in_features = base.weight.shape
        self.A = nn.Parameter(torch.randn(rank, out_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        hidden = self.base(x)
        update = (hidden @ self.A.transpose(0, 1)) @ self.B.transpose(0, 1)
        return hidden + self.scale * update


def adapt_linears(model, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    count = 0
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear):
                setattr(module, name, wrapper(child, rank, alpha))
                count += 1
    return count


def enable_full_attention(model):
    """Let a causal Phase 1 model read tokens on both sides while tagging.

    NER needs the words after a name. The language-model weights stay as
    trained; only the attention mask used for this task is removed.
    """
    count = 0
    for module in model.modules():
        if getattr(module, "causal", None) is not None and hasattr(module, "bidirectional"):
            module.bidirectional = True
            count += 1
    return count


class ChainCRF(nn.Module):
    """Linear-chain CRF over BIO tags. Illegal transitions stay blocked."""

    def __init__(self, tag_names):
        super().__init__()
        n_tags = len(tag_names)
        self.transitions = nn.Parameter(torch.zeros(n_tags, n_tags))
        allowed = torch.zeros(n_tags, n_tags, dtype=torch.bool)
        for src, src_name in enumerate(tag_names):
            for dst, dst_name in enumerate(tag_names):
                if dst_name == "O" or dst_name.startswith("B-"):
                    allowed[src, dst] = True
                else:
                    label = dst_name[2:]
                    allowed[src, dst] = src_name in ("B-" + label, "I-" + label)
        self.register_buffer("allowed", allowed)
        self.register_buffer("score_bias", torch.zeros(n_tags))
        with torch.no_grad():
            self.transitions.masked_fill_(~allowed, -10.0)

    def transition(self):
        return self.transitions.masked_fill(~self.allowed, -1e4)

    def _emit(self, emissions):
        return emissions + self.score_bias.view(1, 1, -1)

    def nll(self, emissions, tags, mask):
        emissions = self._emit(emissions)
        safe = tags.clamp(min=0)
        gold = self._path_score(emissions, safe, mask)
        logz = self._partition(emissions, mask)
        lengths = mask.float().sum(dim=1).clamp(min=1)
        return ((logz - gold) / lengths).mean()

    def _path_score(self, emissions, tags, mask):
        trans = self.transition()
        score = emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for step in range(1, emissions.size(1)):
            jump = trans[tags[:, step - 1], tags[:, step]]
            token = emissions[:, step].gather(1, tags[:, step : step + 1]).squeeze(1)
            score = score + (jump + token) * mask[:, step].float()
        return score

    def _partition(self, emissions, mask):
        trans = self.transition()
        alpha = emissions[:, 0]
        for step in range(1, emissions.size(1)):
            scores = alpha.unsqueeze(2) + trans.unsqueeze(0) + emissions[:, step].unsqueeze(1)
            nxt = torch.logsumexp(scores, dim=1)
            active = mask[:, step].unsqueeze(1)
            alpha = torch.where(active, nxt, alpha)
        return torch.logsumexp(alpha, dim=-1)

    def decode(self, emissions, mask):
        emissions = self._emit(emissions)
        trans = self.transition()
        batch, steps, n_tags = emissions.shape
        back = torch.zeros(batch, steps, n_tags, dtype=torch.long, device=emissions.device)
        alpha = emissions[:, 0]
        for step in range(1, steps):
            scores = alpha.unsqueeze(2) + trans.unsqueeze(0)
            best, index = scores.max(dim=1)
            nxt = best + emissions[:, step]
            active = mask[:, step].unsqueeze(1)
            alpha = torch.where(active, nxt, alpha)
            back[:, step] = index
        lengths = mask.long().sum(dim=1).clamp(min=1)
        tags = torch.zeros(batch, steps, dtype=torch.long, device=emissions.device)
        chosen = alpha.argmax(dim=-1)
        for row in range(batch):
            pos = int(lengths[row]) - 1
            tags[row, pos] = chosen[row]
            for step in range(pos, 0, -1):
                chosen[row] = back[row, step, tags[row, step]]
                tags[row, step - 1] = chosen[row]
        return tags


def hidden_states(model, token_ids, pad_mask):
    """Contextual vector for every token, before the language-model head."""
    if isinstance(model, EncoderDecoderLM):
        return model.encode(token_ids, pad_mask)
    hidden = model.token_plus_position(token_ids)
    hidden = model.run_blocks(model.blocks, hidden, pad_mask)
    return model.norm(hidden)


class TokenTagger(nn.Module):
    def __init__(self, backbone, n_tags, tag_names=None, use_crf=False):
        super().__init__()
        self.backbone = backbone
        width = backbone.embed.weight.size(1)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(width, n_tags)
        nn.init.zeros_(self.head.bias)
        self.head.bias.data[0] = 2.0
        self.crf = ChainCRF(tag_names) if use_crf else None

    def forward(self, token_ids, pad_mask):
        hidden = hidden_states(self.backbone, token_ids, pad_mask)
        return self.head(self.drop(hidden))


def build_tagger(arch, method, vocab_size, n_tags, weights_path, rank, alpha, tag_names=None, use_crf=False):
    backbone = build_model(arch, vocab_size=vocab_size)
    state = torch.load(weights_path, map_location="cpu", weights_only=True)
    backbone.load_state_dict(state)
    if method == "full":
        for parameter in backbone.parameters():
            parameter.requires_grad_(True)
        # NER uses encode() only. The causal decoder is unused, and its
        # gradients do not fit in 6GB next to a wider bidirectional window.
        if arch == "encdec":
            for parameter in backbone.decoder.parameters():
                parameter.requires_grad_(False)
    else:
        kind = "lorma" if method == "lorma" else "lora"
        wrapped = adapt_linears(backbone, kind, rank, alpha)
        print(f"wrapped {wrapped} linear layers with {kind}", flush=True)
    tagger = TokenTagger(backbone, n_tags, tag_names=tag_names, use_crf=use_crf)
    trainable = sum(parameter.numel() for parameter in tagger.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in tagger.parameters())
    print(f"parameters trainable {trainable / 1e6:.2f}M  total {total / 1e6:.2f}M", flush=True)
    return tagger


def class_weights(windows, n_tags, cap):
    counts = torch.ones(n_tags)
    for window in windows:
        for tag in window["target"]:
            counts[tag] += 1
    weights = 1.0 / counts.sqrt()
    weights = weights * (n_tags / weights.sum())
    weights = torch.clamp(weights, max=float(weights.median() * cap))
    return weights


def run_epoch(tagger, windows, optimizer, weights, device, train, batch_size, method, teacher=None, scaler=None):
    if train:
        tagger.train()
        # Frozen Phase 1 layers stay in eval so their dropout does not
        # scramble a 70-document training set. Adapter weights still learn.
        if method != "full":
            tagger.backbone.eval()
    else:
        tagger.eval()
    order = list(windows)
    if train:
        random.shuffle(order)
    total = 0.0
    seen = 0
    step = 0
    optimizer_was_used = optimizer is not None
    if optimizer_was_used:
        optimizer.zero_grad(set_to_none=True)
    for start in range(0, len(order), batch_size):
        chunk = order[start : start + batch_size]
        width = max(len(item["ids"]) for item in chunk)
        token_ids = torch.zeros(len(chunk), width, dtype=torch.long, device=device)
        targets = torch.full((len(chunk), width), -100, dtype=torch.long, device=device)
        pad_mask = torch.zeros(len(chunk), width, dtype=torch.bool, device=device)
        for row, item in enumerate(chunk):
            length = len(item["ids"])
            token_ids[row, :length] = torch.tensor(item["ids"], device=device)
            targets[row, :length] = torch.tensor(item["target"], device=device)
            pad_mask[row, :length] = True
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = tagger(token_ids, pad_mask).float()
                if tagger.crf is None:
                    loss = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        targets.reshape(-1),
                        weight=weights,
                        ignore_index=-100,
                    )
                else:
                    with torch.autocast(device_type=device.type, enabled=False):
                        loss = tagger.crf.nll(logits.float(), targets, pad_mask)
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(token_ids, pad_mask)
                    temperature = 2.0
                    student_log = F.log_softmax(logits / temperature, dim=-1)
                    teacher_prob = F.softmax(teacher_logits / temperature, dim=-1)
                    kl = F.kl_div(student_log, teacher_prob, reduction="none").sum(dim=-1)
                    active = targets.ne(-100)
                    kl = (kl * active).sum() / active.sum().clamp(min=1)
                    loss = loss + (temperature * temperature) * kl
        if train:
            scaled = loss / 1.0
            if scaler is not None:
                scaler.scale(scaled).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(tagger.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                scaled.backward()
                torch.nn.utils.clip_grad_norm_(tagger.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach()) * len(chunk)
        seen += len(chunk)
        step += 1
        if train and step % 50 == 0:
            print(f"  step {step} loss {total / seen:.4f}", flush=True)
    return total / max(1, seen)


def trim_span(text, start, end):
    """Drop punctuation that is glued to a token but is outside the entity.

    The BPE often keeps the comma or sentence period on the last word.
    About 97% of the gold spans are an exact token span once those
    characters are removed. Periods that belong to 'v.' or 'Ltd.' stay.
    """
    while start < end and text[end - 1] in ",;:":
        end -= 1
    while start < end and text[start] == " ":
        start += 1
    if end - start >= 2 and text[end - 1] == "." and text[end - 2].isalnum():
        word = text[start:end]
        keep = word.endswith((" v.", " vs.", " No.", " Ltd.", " Pvt.", " Dr.", " Mr.", " Ms."))
        if not keep:
            end -= 1
    return start, end


def clean_edges(text, spans):
    """Fix citation edges that the tagger consistently gets one or two characters off.

    Held-out judgments keep the period on Ors., Anr. and Cr.P.C., and they
    leave out a glued footnote number, a closing parenthesis, and the words
    'reported in'. Party names are left alone: Ltd. is not a reliable period.
    """
    fixed = []
    for start, end, label in spans:
        if end > start and text[start:end].endswith(")."):
            end -= 2
        while end > start and text[end - 1] in ")]":
            end -= 1
        while end > start + 1 and text[end - 1].isdigit() and text[end - 2] in ".)":
            if end - start >= 5 and text[end - 5 : end - 1].isdigit():
                break
            end -= 1
        folded = text[start:end].casefold()
        marker = folded.find(" reported in ")
        if marker > 0:
            end = start + marker
        if label in ("PREC", "STAT", "CASENO") and end < len(text) and text[end] == ".":
            if text[start:end].endswith(("Ors", "Anr", "Cr.P.C", "I.P.C", "C.P.C")):
                end += 1
        if end > start:
            fixed.append((start, end, label))
    unique = []
    seen = set()
    for span in fixed:
        if span not in seen:
            seen.add(span)
            unique.append(span)
    return unique


def spans_from_tags(text, token_spans, tags, probabilities, thresholds):
    found = []
    index = 0
    while index < len(tags):
        tag = tags[index]
        if not tag.startswith("B-") and not tag.startswith("I-"):
            index += 1
            continue
        label = tag[2:]
        if probabilities[index] < thresholds.get(label, 0.0):
            index += 1
            continue
        cursor = index + 1
        while cursor < len(tags) and tags[cursor] == "I-" + label:
            if probabilities[cursor] < thresholds.get(label, 0.0):
                break
            cursor += 1
        start, end = trim_span(text, token_spans[index][0], token_spans[cursor - 1][1])
        if end > start:
            found.append((start, end, label))
        index = cursor
    # The evaluator rejects a repeated span, so keep the first copy only.
    unique = []
    seen = set()
    for span in found:
        if span not in seen:
            seen.add(span)
            unique.append(span)
    return unique


def propagate_spans(text, spans):
    """Copy a name the model already found onto later exact mentions.

    About two thirds of the gold appellant and respondent spans are the same
    string repeated in one judgment. A window tagger often hits the name once
    and misses the copies. Only labels that are names are copied, and a copy
    that sits inside a span already predicted is left alone.
    """
    occupied = [(start, end) for start, end, _label in spans]
    added = []

    def overlaps(start, end):
        return any(not (end <= left or start >= right) for left, right in occupied)

    def boundary(index):
        return index <= 0 or index >= len(text) or not text[index].isalnum()

    # Respondent, witness, and counsel copies added more wrong spans than
    # right ones on the held-out judgments. Appellant names did the opposite.
    minimum = {"APP": 8}
    folded = text.casefold()
    seen = set()
    for start, end, label in spans:
        if label not in minimum:
            continue
        raw = " ".join(text[start:end].split())
        key = raw.casefold()
        if len(key) < minimum[label] or key in _GENERIC or (label, key) in seen:
            continue
        seen.add((label, key))
        needle = key
        cursor = 0
        while True:
            at = folded.find(needle, cursor)
            if at < 0:
                break
            stop = at + len(needle)
            cursor = at + 1
            if not boundary(at - 1) or not boundary(stop):
                continue
            if overlaps(at, stop):
                continue
            tail = folded[stop : stop + 8]
            if label in ("APP", "RESP") and (
                tail.startswith(" v.") or tail.startswith(" vs") or tail.startswith(" versus")
            ):
                continue
            added.append((at, stop, label))
            occupied.append((at, stop))
    return spans + added


def predict_document(tagger, tokenizer, text, width, stride, tag_names, thresholds, device):
    ids, spans = tokenizer.encode_with_offsets(text)
    if not ids:
        return []
    summed = torch.zeros(len(ids), len(tag_names))
    counts = torch.zeros(len(ids), 1)
    tagger.eval()
    for window in make_windows(ids, spans, ["O"] * len(ids), width, stride):
        token_ids = torch.tensor([window["ids"]], dtype=torch.long, device=device)
        pad_mask = torch.ones(1, len(window["ids"]), dtype=torch.bool, device=device)
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = tagger(token_ids, pad_mask)[0].float().cpu()
        begin = window["offset"]
        summed[begin : begin + len(window["ids"])] += logits
        counts[begin : begin + len(window["ids"])] += 1
    averaged = summed / counts.clamp(min=1)
    if tagger.crf is None:
        probabilities, chosen = averaged.softmax(dim=-1).max(dim=-1)
        chosen = chosen.tolist()
        probabilities = probabilities.tolist()
    else:
        device = next(tagger.parameters()).device
        emissions = averaged.unsqueeze(0).to(device)
        mask = torch.ones(1, averaged.size(0), dtype=torch.bool, device=device)
        chosen = tagger.crf.decode(emissions, mask)[0].cpu().tolist()
        probabilities = averaged.softmax(dim=-1).gather(
            1, torch.tensor(chosen).unsqueeze(1)
        ).squeeze(1).tolist()
    tags = [tag_names[index] for index in chosen]
    found = spans_from_tags(text, spans, tags, probabilities, thresholds)
    if PROPAGATE:
        found = propagate_spans(text, found)
    return clean_edges(text, found)


def macro_f1(gold_rows, predictions):
    stats = {label: [0, 0, 0] for label in LABELS}
    for doc_id, _text, entities in gold_rows:
        gold = {(start, end, label) for start, end, label in entities}
        pred = set(tuple(span) for span in predictions.get(doc_id, []))
        for label in LABELS:
            gold_label = {span for span in gold if span[2] == label}
            pred_label = {span for span in pred if span[2] == label}
            stats[label][0] += len(gold_label & pred_label)
            stats[label][1] += len(pred_label - gold_label)
            stats[label][2] += len(gold_label - pred_label)
    f1s = []
    lines = []
    for label in LABELS:
        true_pos, false_pos, false_neg = stats[label]
        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else 0.0
        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1s.append(f1)
        lines.append(f"{label}: P={precision:.3f} R={recall:.3f} F1={f1:.3f}")
    return sum(f1s) / len(f1s), lines


def score_split(tagger, tokenizer, rows, width, stride, tag_names, thresholds, device):
    predictions = {}
    for index, (doc_id, text, _entities) in enumerate(rows):
        predictions[doc_id] = predict_document(
            tagger, tokenizer, text, width, stride, tag_names, thresholds, device
        )
        if index % 5 == 0:
            print(f"  scored {index + 1}/{len(rows)}", flush=True)
    f1, lines = macro_f1(rows, predictions)
    return f1, lines, predictions


def tune_thresholds(tagger, tokenizer, rows, width, stride, tag_names, device):
    """Pick one minimum probability per label on the held-out documents."""
    _f1, _lines, predictions = score_split(
        tagger, tokenizer, rows, width, stride, tag_names, {label: 0.0 for label in LABELS}, device
    )
    # Re-score from cached spans is not available, so search a short grid by
    # re-decoding is too expensive. The probabilities are already inside
    # predict_document. A second pass with a shared threshold is the stable
    # choice on only a handful of dev documents.
    best_f1 = _f1
    best = {label: 0.0 for label in LABELS}
    print(f"threshold 0.00  macro-F1 {best_f1:.4f}", flush=True)
    for minimum in (0.15, 0.30, 0.45, 0.60, 0.75):
        trial = {label: minimum for label in LABELS}
        f1, _lines, _predictions = score_split(
            tagger, tokenizer, rows, width, stride, tag_names, trial, device
        )
        print(f"threshold {minimum:.2f}  macro-F1 {f1:.4f}", flush=True)
        if f1 > best_f1:
            best_f1 = f1
            best = trial
    return best, best_f1


def split_documents(fold1, fold2, dev_count):
    ordered = sorted(fold2, key=lambda row: row[0])
    dev_rows = ordered[-dev_count:]
    dev_ids = {row[0] for row in dev_rows}
    train_rows = list(fold1) + [row for row in fold2 if row[0] not in dev_ids]
    return train_rows, dev_rows


def collect_windows(tokenizer, rows, width, stride, tag_index):
    windows = []
    for doc_id, text, entities in rows:
        _ids, _spans, _tags, doc_windows = prepare_document(
            tokenizer, doc_id, text, entities, width, stride, tag_index
        )
        windows.extend(doc_windows)
    return windows


def save_checkpoint(path, tagger, args, tag_names, thresholds, dev_f1):
    torch.save(
        {
            "arch": args.arch,
            "method": args.method,
            "tag_names": tag_names,
            "thresholds": thresholds,
            "dev_f1": dev_f1,
            "model": tagger.state_dict(),
        },
        path,
    )
    print("saved", path, flush=True)


def write_submission(path, predictions):
    payload = {doc_id: [list(span) for span in spans] for doc_id, spans in predictions.items()}
    folder = os.path.dirname(path)
    os.makedirs(folder, exist_ok=True)
    pred_path = os.path.join(folder, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False)
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    print("zip", path, "docs", len(payload), flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", required=True, choices=["decoder", "encdec", "hybrid", "efficient"])
    parser.add_argument("--method", required=True, choices=["full", "lora", "lorma", "distill"])
    parser.add_argument("--weights", default=None)
    parser.add_argument("--bpe", default=DEFAULT_BPE)
    parser.add_argument("--teacher", default=None, help="Checkpoint from a larger Phase 2 model.")
    parser.add_argument("--teacher_arch", default="efficient")
    parser.add_argument("--teacher_method", default="lorma")
    parser.add_argument("--teacher_weights", default=None)
    parser.add_argument("--out", default=None)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch", type=int, default=0)
    parser.add_argument("--width", type=int, default=0)
    parser.add_argument("--stride", type=int, default=0)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=float, default=16.0)
    parser.add_argument("--lr", type=float, default=0.0)
    parser.add_argument("--head_lr", type=float, default=0.0, help="Learning rate for the tag head and CRF.")
    parser.add_argument("--dev_count", type=int, default=8)
    parser.add_argument("--weight_cap", type=float, default=5.0)
    parser.add_argument("--crf", action="store_true")
    parser.add_argument("--bidirectional", action="store_true")
    parser.add_argument("--data_dir", default=None, help="Local Phase_2_LNER folder, skipping Hugging Face.")
    parser.add_argument("--check", action="store_true")
    parser.add_argument("--resume", default=None, help="Phase 2 checkpoint to continue from.")
    parser.add_argument("--unfreeze", action="store_true", help="Train the Phase 1 weights under the adapter.")
    parser.add_argument("--score_only", action="store_true", help="Score the resume checkpoint and write a zip.")
    parser.add_argument("--seed", type=int, default=SEED)
    args = parser.parse_args()
    global DATA_DIR
    DATA_DIR = args.data_dir

    if args.method == "full":
        args.batch = args.batch or 1
        args.width = args.width or 256
        args.stride = args.stride or 128
        args.lr = args.lr or 2e-5
    else:
        args.batch = args.batch or 2
        args.width = args.width or 384
        args.stride = args.stride or 192
        args.lr = args.lr or 1e-4
    if args.weights is None:
        args.weights = DEFAULT_WEIGHTS[args.arch]
    if args.out is None:
        args.out = os.path.join(LOCAL, "phase2_runs", f"{args.arch}_{args.method}")
    os.makedirs(args.out, exist_ok=True)

    set_seed(args.seed)
    print("seed", args.seed, flush=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device", device, flush=True)
    if not os.path.isfile(args.weights):
        raise SystemExit(f"missing Phase 1 weights: {args.weights}")
    if not os.path.isfile(args.bpe):
        raise SystemExit(f"missing BPE: {args.bpe}")

    tokenizer = LegalBPE.load(args.bpe)
    fold1 = load_fold("Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json")
    fold2 = load_fold("Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json")
    print("fold1", len(fold1), "fold2", len(fold2), flush=True)
    covered, total = boundary_report(tokenizer, fold1 + fold2)
    if total:
        print(f"exact token-boundary rate {covered / total:.3f}", flush=True)
    if args.check:
        return

    train_rows, dev_rows = split_documents(fold1, fold2, args.dev_count)
    print("train docs", len(train_rows), "dev docs", [row[0] for row in dev_rows], flush=True)
    names = tag_names_of()
    tag_index = {name: index for index, name in enumerate(names)}
    train_windows = collect_windows(tokenizer, train_rows, args.width, args.stride, tag_index)
    print("train windows", len(train_windows), "width", args.width, "stride", args.stride, flush=True)

    tagger = build_tagger(
        args.arch, args.method, len(tokenizer), len(names), args.weights, args.rank, args.alpha, names, args.crf
    )
    if args.bidirectional:
        opened = enable_full_attention(tagger.backbone)
        print("full attention layers", opened, flush=True)
    best_path = os.path.join(args.out, "best.pt")
    best_f1 = -1.0
    if args.resume:
        resumed = torch.load(args.resume, map_location="cpu", weights_only=True)
        tagger.load_state_dict(resumed["model"])
        best_f1 = float(resumed.get("dev_f1", -1.0))
        print("resumed", args.resume, "dev macro-F1", round(best_f1, 4), flush=True)
    if args.unfreeze:
        opened = 0
        for module in tagger.modules():
            if isinstance(module, (LoRMALinear, LoRALinear)):
                module.base.weight.requires_grad_(True)
                if module.base.bias is not None:
                    module.base.bias.requires_grad_(True)
                opened += 1
        print("unfroze", opened, "adapter base matrices", flush=True)
    tagger = tagger.to(device)
    teacher = None
    if args.method == "distill":
        if not args.teacher:
            raise SystemExit("distill needs --teacher pointing at a Phase 2 checkpoint")
        teacher_weights = args.teacher_weights or DEFAULT_WEIGHTS[args.teacher_arch]
        checkpoint = torch.load(args.teacher, map_location="cpu", weights_only=True)
        teacher_state = checkpoint["model"]
        teacher_crf = any(key.startswith("crf.") for key in teacher_state)
        teacher = build_tagger(
            args.teacher_arch,
            args.teacher_method,
            len(tokenizer),
            len(names),
            teacher_weights,
            args.rank,
            args.alpha,
            names,
            teacher_crf,
        )
        teacher.load_state_dict(teacher_state)
        if args.bidirectional:
            opened = enable_full_attention(teacher.backbone)
            print("teacher full attention layers", opened, flush=True)
        teacher = teacher.to(device).eval()
        for parameter in teacher.parameters():
            parameter.requires_grad_(False)

    weights = class_weights(train_windows, len(names), args.weight_cap).to(device)
    if tagger.crf is not None:
        tagger.crf.score_bias.copy_(weights.log().to(tagger.crf.score_bias.device))
    if args.score_only:
        global PROPAGATE
        zero = {label: 0.0 for label in LABELS}
        PROPAGATE = False
        base_f1, base_lines, _base_pred = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, zero, device
        )
        print(f"base dev macro-F1 {base_f1:.4f}", flush=True)
        print("\n".join(base_lines), flush=True)
        PROPAGATE = True
        prop_f1, prop_lines, _prop_pred = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, zero, device
        )
        print(f"propagate dev macro-F1 {prop_f1:.4f}", flush=True)
        print("\n".join(prop_lines), flush=True)
        PROPAGATE = prop_f1 > base_f1 + 1e-4
        print("zip uses propagate" if PROPAGATE else "zip uses the model spans only", flush=True)
        test_rows = load_fold("Phase_2_LNER/Test/fold3-text.json")
        predictions = {}
        for index, (doc_id, text, _entities) in enumerate(test_rows):
            predictions[doc_id] = predict_document(
                tagger, tokenizer, text, args.width, args.stride, names, zero, device
            )
            print(f"test {index + 1}/{len(test_rows)} id {doc_id} spans {len(predictions[doc_id])}", flush=True)
        write_submission(os.path.join(args.out, "submission.zip"), predictions)
        return
    head_params = []
    backbone_params = []
    for name, parameter in tagger.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("head.") or name.startswith("crf."):
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)
    if args.head_lr:
        optimizer = torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": args.lr},
                {"params": head_params, "lr": args.head_lr},
            ],
            weight_decay=0.01,
        )
        print(f"lr backbone {args.lr} head {args.head_lr}", flush=True)
    else:
        optimizer = torch.optim.AdamW(backbone_params + head_params, lr=args.lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    if args.resume and not os.path.isfile(best_path):
        shutil.copyfile(args.resume, best_path)

    patience = 0
    for epoch in range(args.epochs):
        loss = run_epoch(
            tagger, train_windows, optimizer, weights, device, True, args.batch, args.method, teacher, scaler
        )
        print(f"epoch {epoch + 1} train loss {loss:.4f}", flush=True)
        f1, lines, _predictions = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, {label: 0.0 for label in LABELS}, device
        )
        print(f"epoch {epoch + 1} dev macro-F1 {f1:.4f}", flush=True)
        print("\n".join(lines), flush=True)
        if f1 > best_f1:
            best_f1 = f1
            patience = 0
            save_checkpoint(best_path, tagger, args, names, {label: 0.0 for label in LABELS}, f1)
        else:
            patience += 1
            if patience >= 3:
                print("early stop", flush=True)
                break

    checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
    tagger.load_state_dict(checkpoint["model"])
    tagger.to(device)
    thresholds, tuned_f1 = tune_thresholds(
        tagger, tokenizer, dev_rows, args.width, args.stride, names, device
    )
    print("chosen thresholds", thresholds, "dev macro-F1", round(tuned_f1, 4), flush=True)
    save_checkpoint(best_path, tagger, args, names, thresholds, tuned_f1)

    test_rows = load_fold("Phase_2_LNER/Test/fold3-text.json")
    predictions = {}
    for index, (doc_id, text, _entities) in enumerate(test_rows):
        predictions[doc_id] = predict_document(
            tagger, tokenizer, text, args.width, args.stride, names, thresholds, device
        )
        print(f"test {index + 1}/{len(test_rows)} id {doc_id} spans {len(predictions[doc_id])}", flush=True)
    write_submission(os.path.join(args.out, "submission.zip"), predictions)


In [ ]:
_register('phase2_lner')


In [ ]:
"""Phase 4 court-judgment prediction from the Phase 1 models.

Each case is read as a few token windows spread through the document.
The Phase 1 model stays frozen except for the chosen adaptation:
LoRMA, BitFit, a bottleneck adapter, the last blocks, or distillation.

    python phase4_own.py --arch efficient --method lorma --out runs/eff_lorma
"""
import argparse
import json
import math
import os
import random
import time
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F

from legal_bpe import LegalBPE
from phase1_models import build_model
from phase2_lner import LoRALinear, adapt_linears, enable_full_attention, hidden_states

LOCAL = r"C:\Users\Anshuman Raj\CS781"
USE_FP16 = True
DEFAULT_WEIGHTS = {
    "efficient": os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\model.pt"),
    "decoder": os.path.join(LOCAL, r"local_weights\cs781_phase1_decoder\model.pt"),
    "hybrid": os.path.join(LOCAL, r"local_weights\cs781_phase1_hybrid\model.pt"),
    "encdec": os.path.join(LOCAL, r"encdec_run2\cs781_phase1_encdec\model.pt"),
}
DEFAULT_BPE = os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\legal_bpe.json")
DEFAULT_DATA = os.path.join(LOCAL, "phase4data")


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_rows(path, labeled):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    rows = []
    for item in payload:
        text = item.get("text") or ""
        label = item.get("label") if labeled else None
        if labeled and label not in (0, 1):
            continue
        if not str(text).strip():
            continue
        rows.append({"id": str(item["id"]), "text": text, "label": None if label is None else int(label)})
    return rows


def shorten(text, word_cap, focus="ends"):
    words = text.split()
    if len(words) <= word_cap:
        return text
    if focus == "tail":
        return " ".join(words[-word_cap:])
    head = word_cap * 2 // 3
    return " ".join(words[:head] + words[-(word_cap - head) :])


def windows_of(ids, count, width, cover="spread"):
    if not ids:
        return [[1]]
    if cover == "tail":
        ids = ids[-(count * width) :]
        parts = [ids[start : start + width] for start in range(0, len(ids), width)]
        return parts or [[1]]
    if len(ids) <= width:
        return [ids]
    if len(ids) <= count * width:
        return [ids[start : start + width] for start in range(0, len(ids), width)][:count]
    span = len(ids) - width
    starts = [int(round(i * span / (count - 1))) for i in range(count)]
    return [ids[start : start + width] for start in starts]


class BlockAdapter(nn.Module):
    """Frozen block plus a bottleneck residual, starting at zero."""

    def __init__(self, block, width, rank):
        super().__init__()
        self.block = block
        for parameter in block.parameters():
            parameter.requires_grad_(False)
        self.down = nn.Linear(width, rank)
        self.up = nn.Linear(rank, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x, pad_mask=None):
        hidden = self.block(x, pad_mask=pad_mask)
        return hidden + self.up(F.relu(self.down(hidden)))


def add_adapters(blocks, width, rank):
    for index, block in enumerate(list(blocks)):
        blocks[index] = BlockAdapter(block, width, rank)
    return len(blocks)


def apply_method(model, arch, method, rank, alpha, unfreeze=2):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    blocks = model.encoder if arch == "encdec" else model.blocks
    if method in ("lorma", "lora"):
        wrapped = adapt_linears(blocks, "lorma" if method == "lorma" else "lora", rank, alpha)
        print(f"wrapped {wrapped} linear layers with {method}", flush=True)
    elif method == "adapter":
        width = model.embed.weight.size(1)
        print(f"added adapters on {add_adapters(blocks, width, rank)} blocks", flush=True)
    elif method == "bitfit":
        count = 0
        for name, parameter in model.named_parameters():
            if name.endswith("bias"):
                parameter.requires_grad_(True)
                count += parameter.numel()
        print(f"bitfit bias parameters {count}", flush=True)
    elif method == "partial":
        chosen = list(blocks)[-max(1, unfreeze) :]
        count = 0
        for block in chosen:
            for parameter in block.parameters():
                parameter.requires_grad_(True)
                count += parameter.numel()
        print(f"unfroze last {len(chosen)} blocks, {count} parameters", flush=True)
    else:
        raise SystemExit(f"unknown method {method}")


class CaseClassifier(nn.Module):
    def __init__(self, backbone, arch, extra=0):
        super().__init__()
        self.backbone = backbone
        self.arch = arch
        self.extra = extra
        width = backbone.embed.weight.size(1)
        self.token_score = nn.Linear(width, 1)
        self.chunk_score = nn.Linear(width, 1)
        self.end_bias = nn.Parameter(torch.tensor(2.0))
        self.head = nn.Linear(width + extra, 2)
        nn.init.zeros_(self.token_score.weight)
        nn.init.zeros_(self.token_score.bias)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        if extra:
            with torch.no_grad():
                self.head.weight[0, -1] = -1.0
                self.head.weight[1, -1] = 1.0

    def encode_windows(self, windows, device):
        order = sorted(range(len(windows)), key=lambda index: len(windows[index]))
        vectors = [None] * len(windows)
        start = 0
        while start < len(order):
            stop = start + 1
            while stop < len(order) and (stop - start + 1) * len(windows[order[stop]]) <= 4096:
                stop += 1
            chunk = order[start:stop]
            width = max(len(windows[index]) for index in chunk)
            token_ids = torch.zeros(len(chunk), width, dtype=torch.long, device=device)
            pad_mask = torch.zeros(len(chunk), width, dtype=torch.bool, device=device)
            for row, index in enumerate(chunk):
                ids = windows[index]
                token_ids[row, : len(ids)] = torch.tensor(ids, device=device)
                pad_mask[row, : len(ids)] = True
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16 and device.type == "cuda"):
                hidden = hidden_states(self.backbone, token_ids, pad_mask)
            score = self.token_score(hidden.float()).squeeze(-1)
            positions = torch.arange(width, device=device, dtype=torch.float32) / max(width - 1, 1)
            score = score + self.end_bias * positions.unsqueeze(0)
            score = score.masked_fill(~pad_mask, -1e4)
            weight = torch.softmax(score, dim=-1).unsqueeze(-1)
            pooled = (hidden.float() * weight).sum(dim=1)
            for row, index in enumerate(chunk):
                vectors[index] = pooled[row]
            start = stop
        return torch.stack(vectors)

    def logits_many(self, batch_windows, device, extras=None):
        flat = [window for windows in batch_windows for window in windows]
        vectors = self.encode_windows(flat, device)
        if extras is None:
            extras = torch.zeros(len(batch_windows), self.extra, device=device)
        outputs = []
        start = 0
        for row_index, windows in enumerate(batch_windows):
            stop = start + len(windows)
            picked = vectors[start:stop]
            mix = torch.softmax(self.chunk_score(picked).squeeze(-1), dim=0).unsqueeze(-1)
            pooled = (mix * picked).sum(dim=0)
            if self.extra:
                pooled = torch.cat([pooled, extras[row_index].to(pooled.device)], dim=0)
            outputs.append(self.head(pooled))
            start = stop
        return torch.stack(outputs)

    def logits(self, windows, device):
        return self.logits_many([windows], device)[0]


def macro_f1(gold, pred):
    scores = []
    for label in (0, 1):
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        if tp == 0:
            scores.append(0.0)
            continue
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        scores.append(2 * precision * recall / (precision + recall))
    return sum(scores) / 2


def sentence_windows(text, count, width, tokenizer):
    parts = []
    for piece in str(text).replace("!", ".").replace("?", ".").split("."):
        if len(piece.split()) >= 4:
            parts.append(piece.strip())
    windows = []
    for part in parts[-count:]:
        ids = tokenizer.encode(part)
        if len(ids) > width:
            ids = ids[-width:]
        if ids:
            windows.append(ids)
    return windows or [[1]]


def prepare(rows, tokenizer, chunks, width, word_cap, focus="ends", cover="spread", tail_tokens=0, sentences=0):
    prepared = []
    for row in rows:
        if sentences:
            windows = sentence_windows(row["text"], sentences, width, tokenizer)
        else:
            ids = tokenizer.encode(shorten(row["text"], word_cap, focus))
            if tail_tokens:
                windows = [ids[-tail_tokens:] or [1]]
            else:
                windows = windows_of(ids, chunks, width, cover)
        item = {
            "id": row["id"],
            "label": row["label"],
            "windows": windows,
        }
        if "word_prob" in row:
            item["word_prob"] = float(row["word_prob"])
        prepared.append(item)
    return prepared


def batch_extras(batch, device):
    if "word_prob" not in batch[0]:
        return None
    return torch.tensor([[row["word_prob"]] for row in batch], dtype=torch.float32, device=device)


def positive_scores(model, rows, device, batch_size=16):
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start : start + batch_size]
            logits = model.logits_many(
                [row["windows"] for row in batch], device, batch_extras(batch, device)
            )
            scores.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    return scores


def best_cut(gold, positive):
    best_f1, best_threshold = -1.0, 0.5
    for cut in [index / 100 for index in range(15, 86)]:
        pred = [1 if score >= cut else 0 for score in positive]
        f1 = macro_f1(gold, pred)
        if f1 > best_f1:
            best_f1, best_threshold = f1, cut
    return best_threshold, best_f1


def balanced_indices(rows):
    groups = {0: [], 1: []}
    for index, row in enumerate(rows):
        groups[int(row["label"])].append(index)
    width = max(len(groups[0]), len(groups[1]), 1)
    order = []
    for label in (0, 1):
        pool = groups[label] or [0]
        picked = [pool[index % len(pool)] for index in range(width)]
        random.shuffle(picked)
        order.extend(picked)
    random.shuffle(order)
    return order


def set_lrs(optimizer, step, total_steps, warmup, bases):
    if total_steps <= 1 or warmup <= 0 and step >= total_steps:
        scale = 1.0
    elif step < warmup:
        scale = (step + 1) / max(1, warmup)
    else:
        progress = (step - warmup) / max(1, total_steps - warmup)
        scale = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    for group, base in zip(optimizer.param_groups, bases):
        group["lr"] = base * scale


def dump_scores(path, rows, scores):
    payload = [{"id": row["id"], "score": round(float(score), 6)} for row, score in zip(rows, scores)]
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle)


def predict(model, rows, device, batch_size=16):
    model.eval()
    guesses = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start : start + batch_size]
            logits = model.logits_many(
                [row["windows"] for row in batch], device, batch_extras(batch, device)
            )
            guesses.extend(logits.argmax(dim=-1).tolist())
    return guesses


def teacher_logits(args, rows, device):
    checkpoint = torch.load(args.teacher, map_location="cpu", weights_only=False)
    teacher = build_case(checkpoint["arch"], checkpoint["method"], checkpoint["weights"], checkpoint["rank"], checkpoint["alpha"])
    teacher.load_state_dict(checkpoint["model"], strict=False)
    teacher.to(device)
    teacher.eval()
    print("teacher", checkpoint["arch"], checkpoint["method"], "dev", round(checkpoint["dev_f1"], 4), flush=True)
    scores = []
    with torch.no_grad():
        for row in rows:
            scores.append(teacher.logits(row["windows"], device).float().cpu())
    del teacher
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return scores


def build_case(arch, method, weights, rank, alpha, unfreeze=2, extra=0):
    backbone = build_model(arch, vocab_size=8192)
    backbone.load_state_dict(torch.load(weights, map_location="cpu", weights_only=True))
    enable_full_attention(backbone)
    apply_method(backbone, arch, method, rank, alpha, unfreeze)
    return CaseClassifier(backbone, arch, extra)


def trainable_state(model):
    return {name: value.detach().cpu() for name, value in model.state_dict().items() if _keep(model, name)}


def _keep(model, name):
    if not name.startswith("backbone."):
        return True
    parameter_name = name.replace(".weight", "").replace(".bias", "")
    # Keep every tensor whose matching parameter is trainable, plus adapter masks.
    named = dict(model.named_parameters())
    if name in named and named[name].requires_grad:
        return True
    return name.endswith(".mask")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", choices=("efficient", "decoder", "hybrid", "encdec"), default="efficient")
    parser.add_argument("--method", choices=("lorma", "lora", "bitfit", "adapter", "partial", "distill"), default="lorma")
    parser.add_argument("--student_method", choices=("lorma", "lora", "adapter", "bitfit", "partial"), default="lora")
    parser.add_argument("--teacher", default=None)
    parser.add_argument("--weights", default=None)
    parser.add_argument("--bpe", default=DEFAULT_BPE)
    parser.add_argument("--data_dir", default=DEFAULT_DATA)
    parser.add_argument("--out", required=True)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--max_docs", type=int, default=3000)
    parser.add_argument("--chunks", type=int, default=4)
    parser.add_argument("--chunk_tokens", type=int, default=96)
    parser.add_argument("--word_cap", type=int, default=6000)
    parser.add_argument("--focus", choices=("ends", "tail"), default="ends")
    parser.add_argument("--cover", choices=("spread", "tail"), default="spread")
    parser.add_argument("--unfreeze", type=int, default=2)
    parser.add_argument("--word_feature", action="store_true")
    parser.add_argument("--tail_tokens", type=int, default=0)
    parser.add_argument("--batch", type=int, default=8)
    parser.add_argument("--balanced", action="store_true")
    parser.add_argument("--warmup", type=int, default=0)
    parser.add_argument("--label_smoothing", type=float, default=0.0)
    parser.add_argument("--unfreeze_embed", action="store_true")
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--head_lr", type=float, default=1e-3)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=float, default=16.0)
    parser.add_argument("--kd_weight", type=float, default=0.5)
    parser.add_argument("--kd_temp", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max_steps", type=int, default=0)
    parser.add_argument("--resume", default="")
    parser.add_argument("--sentences", type=int, default=0)
    parser.add_argument("--eval_every", type=int, default=0)
    parser.add_argument("--skip_docs", type=int, default=0)
    parser.add_argument("--train_dropout", action="store_true")
    args = parser.parse_args()
    if args.method == "distill" and not args.teacher:
        raise SystemExit("distill needs --teacher")
    method = args.student_method if args.method == "distill" else args.method
    args.weights = args.weights or DEFAULT_WEIGHTS[args.arch]
    if method == "bitfit":
        args.lr = max(args.lr, 1e-3)
    if method == "partial":
        args.lr = min(args.lr, 1e-4)

    set_seed(args.seed)
    os.makedirs(args.out, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device", device, "arch", args.arch, "method", args.method, "adapter", method, flush=True)

    root = os.path.join(args.data_dir, "Phase_4_CJPE")
    train_rows = read_rows(os.path.join(root, "Train", "train.json"), True)
    dev_rows = read_rows(os.path.join(root, "Train", "dev.json"), True)
    test_rows = read_rows(os.path.join(root, "Test", "test.json"), False)
    print("docs", len(train_rows), len(dev_rows), len(test_rows), flush=True)
    extra = 1 if args.word_feature else 0
    if args.word_feature:
        from phase4_words import fit_word_model, word_probabilities

        saved = os.path.join(LOCAL, "phase4_runs", "word_end.joblib")
        word_path = saved if os.path.isfile(saved) else os.path.join(args.out, "word_end.joblib")
        bundle = fit_word_model(train_rows, word_path)
        for split in (train_rows, dev_rows, test_rows):
            probs = word_probabilities(bundle, [row["text"] for row in split])
            for row, prob in zip(split, probs):
                row["word_prob"] = float(prob)
        print("word feature on", flush=True)
    random.shuffle(train_rows)
    if args.skip_docs:
        train_rows = train_rows[args.skip_docs :]
        print("skipped", args.skip_docs, "already used docs", flush=True)
    train_rows = train_rows[: args.max_docs]
    counts = [sum(row["label"] == label for row in train_rows) for label in (0, 1)]
    if args.balanced:
        weights = torch.ones(2, device=device)
    else:
        weights = torch.tensor([len(train_rows) / (2 * max(1, count)) for count in counts], device=device)
    print("train subset", len(train_rows), "class weights", [round(float(value), 3) for value in weights], flush=True)

    tokenizer = LegalBPE.load(args.bpe)
    started = time.time()
    train_rows = prepare(
        train_rows, tokenizer, args.chunks, args.chunk_tokens, args.word_cap, args.focus, args.cover, args.tail_tokens, args.sentences
    )
    dev_rows = prepare(
        dev_rows, tokenizer, args.chunks, args.chunk_tokens, args.word_cap, args.focus, args.cover, args.tail_tokens, args.sentences
    )
    test_rows = prepare(
        test_rows, tokenizer, args.chunks, args.chunk_tokens, args.word_cap, args.focus, args.cover, args.tail_tokens, args.sentences
    )
    print(
        "focus", args.focus, "cover", args.cover, "chunks", args.chunks,
        "width", args.chunk_tokens, "tail", args.tail_tokens, flush=True,
    )
    print(f"tokenized {time.time() - started:.0f}s", flush=True)

    model = build_case(args.arch, method, args.weights, args.rank, args.alpha, args.unfreeze, extra).to(device)
    if args.unfreeze_embed:
        model.backbone.embed.weight.requires_grad_(True)
        print("unfroze token embeddings", flush=True)
    soft = teacher_logits(args, train_rows, device) if args.method == "distill" else None

    head = [parameter for name, parameter in model.named_parameters() if not name.startswith("backbone.")]
    body = [parameter for name, parameter in model.named_parameters() if name.startswith("backbone.") and parameter.requires_grad]
    groups = [{"params": head, "lr": args.head_lr}]
    if body:
        groups.append({"params": body, "lr": args.lr})
    print(
        f"trainable head {sum(p.numel() for p in head)/1e6:.3f}M  body {sum(p.numel() for p in body)/1e6:.3f}M",
        flush=True,
    )
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    base_lrs = [group["lr"] for group in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    best_f1 = -1.0
    best_path = os.path.join(args.out, "best.pt")
    if args.resume:
        prior = torch.load(args.resume, map_location="cpu", weights_only=False)
        missing, unexpected = model.load_state_dict(prior["model"], strict=False)
        best_f1 = float(prior.get("dev_f1", -1.0))
        torch.save(prior, best_path)
        print(
            f"resumed dev {best_f1:.4f} missing {len(missing)} unexpected {len(unexpected)}",
            flush=True,
        )
    steps_per_epoch = math.ceil(len(train_rows) / max(1, args.batch))
    if args.balanced:
        steps_per_epoch = math.ceil((2 * max(counts)) / max(1, args.batch))
    total_steps = max(1, steps_per_epoch * args.epochs)

    for epoch in range(1, args.epochs + 1):
        started = time.time()
        model.train()
        if not args.train_dropout:
            for module in model.backbone.modules():
                if isinstance(module, nn.Dropout):
                    module.eval()
        order = balanced_indices(train_rows) if args.balanced else list(range(len(train_rows)))
        if not args.balanced:
            random.shuffle(order)
        total = 0.0
        recent = 0.0
        step = 0
        batch_size = args.batch
        for start in range(0, len(order), batch_size):
            step += 1
            global_step = (epoch - 1) * steps_per_epoch + step
            set_lrs(optimizer, global_step, total_steps, args.warmup, base_lrs)
            chosen = order[start : start + batch_size]
            batch = [train_rows[index] for index in chosen]
            logits = model.logits_many(
                [row["windows"] for row in batch], device, batch_extras(batch, device)
            )
            target = torch.tensor([row["label"] for row in batch], device=device)
            loss = F.cross_entropy(logits, target, weight=weights, label_smoothing=args.label_smoothing)
            if soft is not None:
                teacher = torch.stack([soft[index] for index in chosen]).to(device)
                log_p = F.log_softmax(logits / args.kd_temp, dim=-1)
                q = F.softmax(teacher / args.kd_temp, dim=-1)
                loss = (1 - args.kd_weight) * loss + args.kd_weight * (args.kd_temp ** 2) * F.kl_div(
                    log_p, q, reduction="batchmean"
                )
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(head + body, 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += loss.item()
            recent += loss.item()
            if step == 1:
                print(f"  grad norm {float(grad_norm):.4f}", flush=True)
            if args.max_steps and step >= args.max_steps:
                break
            if step % 50 == 0:
                print(
                    f"  epoch {epoch} step {step}/{len(order)} loss {recent / 50:.4f} {time.time() - started:.0f}s",
                    flush=True,
                )
                recent = 0.0
            if args.eval_every and step % args.eval_every == 0:
                dev_scores = positive_scores(model, dev_rows, device)
                gold = [row["label"] for row in dev_rows]
                step_f1 = macro_f1(gold, [1 if score >= 0.5 else 0 for score in dev_scores])
                print(f"  dev argmax {step_f1:.4f} at step {step}", flush=True)
                if step_f1 > best_f1:
                    best_f1 = step_f1
                    torch.save(
                        {
                            "arch": args.arch,
                            "method": method,
                            "rank": args.rank,
                            "alpha": args.alpha,
                            "weights": args.weights,
                            "dev_f1": step_f1,
                            "model": trainable_state(model),
                        },
                        best_path,
                    )
                    print("saved", best_path, flush=True)
                model.train()
                for module in model.backbone.modules():
                    if isinstance(module, nn.Dropout):
                        module.eval()
        dev_scores = positive_scores(model, dev_rows, device)
        gold = [row["label"] for row in dev_rows]
        argmax_f1 = macro_f1(gold, [1 if score >= 0.5 else 0 for score in dev_scores])
        _, tuned_f1 = best_cut(gold, dev_scores)
        print(
            f"epoch {epoch} loss {total / max(1, step):.4f} dev argmax {argmax_f1:.4f} tuned {tuned_f1:.4f} {time.time() - started:.0f}s",
            flush=True,
        )
        if argmax_f1 > best_f1:
            best_f1 = argmax_f1
            torch.save(
                {
                    "arch": args.arch,
                    "method": method,
                    "rank": args.rank,
                    "alpha": args.alpha,
                    "weights": args.weights,
                    "dev_f1": argmax_f1,
                    "model": trainable_state(model),
                },
                best_path,
            )
            print("saved", best_path, flush=True)

    checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
    model = build_case(args.arch, method, args.weights, args.rank, args.alpha, args.unfreeze, extra)
    model.load_state_dict(checkpoint["model"], strict=False)
    model.to(device)
    dev_scores = positive_scores(model, dev_rows, device)
    gold = [row["label"] for row in dev_rows]
    argmax_f1 = macro_f1(gold, [1 if score >= 0.5 else 0 for score in dev_scores])
    cut, tuned_f1 = best_cut(gold, dev_scores)
    # Argmax tracks the hidden test score. A wild dev threshold does not.
    use_cut = cut if tuned_f1 >= argmax_f1 + 0.015 and 0.35 <= cut <= 0.65 else 0.5
    chosen_f1 = tuned_f1 if use_cut != 0.5 else argmax_f1
    print(f"best dev macro-F1 {chosen_f1:.4f} threshold {use_cut:.2f} argmax {argmax_f1:.4f}", flush=True)
    test_scores = positive_scores(model, test_rows, device)
    dump_scores(os.path.join(args.out, "dev_scores.json"), dev_rows, dev_scores)
    dump_scores(os.path.join(args.out, "test_scores.json"), test_rows, test_scores)
    with open(os.path.join(args.out, "metrics.json"), "w", encoding="utf-8") as handle:
        json.dump(
            {"argmax": argmax_f1, "tuned": tuned_f1, "threshold": use_cut, "chosen": chosen_f1},
            handle,
        )
    predictions = [
        {"id": row["id"], "pred": int(score >= use_cut)} for row, score in zip(test_rows, test_scores)
    ]
    pred_path = os.path.join(args.out, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, ensure_ascii=False)
    zip_path = os.path.join(args.out, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    print("zip", zip_path, "rows", len(predictions), flush=True)


In [ ]:
_register('phase4_own')


## 3. Extractive selector (selected)


In [ ]:
"""Phase 6 abstractive summarization from the four Phase 1 models.

Causal models (decoder, hybrid, efficient) do SFT: document + BOS + summary,
loss only on the summary tokens (Module 3.1). Encoder-decoder encodes the
document and decodes the summary. Do not use EncoderDecoderLM.forward — that
split was for Phase 1 language modeling.

    python phase6_summ.py --arch decoder --method lorma --out ... --smoke --require_cuda
"""
import argparse
import json
import math
import os
import random
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import re

from legal_bpe import BOS_ID, EOS_ID, PAD_ID, LegalBPE
from phase1_models import EncoderDecoderLM, BLOCK, build_model, generate, sample_top_p
from phase2_lner import adapt_linears, hidden_states
from phase4_own import DEFAULT_BPE, DEFAULT_WEIGHTS, LOCAL, add_adapters, set_lrs, set_seed

DEFAULT_DATA = os.path.join(LOCAL, "phase6data")


class CrossAdapter(nn.Module):
    """Houlsby-style bottleneck on an encoder-decoder cross-attention block."""

    def __init__(self, block, width, rank):
        super().__init__()
        self.block = block
        for parameter in block.parameters():
            parameter.requires_grad_(False)
        self.down = nn.Linear(width, rank)
        self.up = nn.Linear(rank, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x, memory, pad_mask=None, memory_mask=None):
        hidden = self.block(x, memory, pad_mask=pad_mask, memory_mask=memory_mask)
        return hidden + self.up(F.relu(self.down(hidden)))


def read_rows(path, labeled):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    rows = []
    if isinstance(payload, dict):
        first = next(iter(payload.values())) if payload else None
        if first is not None and not isinstance(first, dict):
            for key, value in payload.items():
                rows.append({"id": str(key), "document": str(value), "summary": ""})
            return [row for row in rows if row["id"] and row["document"].strip()]
        items = list(payload.values())
    else:
        items = payload
    for item in items:
        if not isinstance(item, dict):
            continue
        doc = item.get("document") or item.get("text") or ""
        summary = str(item.get("summary") or "") if labeled else ""
        rows.append({"id": str(item.get("id", "")), "document": str(doc), "summary": summary})
    return [row for row in rows if row["id"] and row["document"].strip()]


def clip_ids(ids, budget):
    if len(ids) <= budget:
        return list(ids)
    head = max(1, budget * 2 // 3)
    tail = budget - head
    return list(ids[:head] + ids[-tail:])


def pack_causal(src, tgt, src_budget, tgt_budget):
    src = clip_ids(src, src_budget)
    tgt = list(tgt[:tgt_budget])
    seq = src + [BOS_ID] + tgt + [EOS_ID]
    labels = [-100] * len(src) + [-100] + tgt + [EOS_ID]
    return seq, labels


def rouge_n(pred, gold):
    pred_t, gold_t = pred.lower().split(), gold.lower().split()
    if not pred_t and not gold_t:
        return 1.0
    if not pred_t or not gold_t:
        return 0.0
    from collections import Counter
    overlap = sum((Counter(pred_t) & Counter(gold_t)).values())
    precision, recall = overlap / len(pred_t), overlap / len(gold_t)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def rouge_l(pred, gold):
    left, right = pred.lower().split(), gold.lower().split()
    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0
    prev = [0] * (len(right) + 1)
    for token in left:
        cur = [0]
        for index, other in enumerate(right, 1):
            cur.append(prev[index - 1] + 1 if token == other else max(prev[index], cur[-1]))
        prev = cur
    overlap = prev[-1]
    precision, recall = overlap / len(left), overlap / len(right)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def adapt(model, arch, method, rank, alpha, unfreeze):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    if arch == "encdec":
        groups = [model.encoder, model.decoder]
    else:
        groups = [model.blocks]
    if method in ("lorma", "lora"):
        wrapped = 0
        for group in groups:
            wrapped += adapt_linears(group, "lorma" if method == "lorma" else "lora", rank, alpha)
        print(f"wrapped {wrapped} linear layers with {method}", flush=True)
    elif method == "adapter":
        width = model.embed.weight.size(1)
        count = 0
        if arch == "encdec":
            count += add_adapters(model.encoder, width, rank)
            for index, block in enumerate(list(model.decoder)):
                model.decoder[index] = CrossAdapter(block, width, rank)
                count += 1
        else:
            count += add_adapters(model.blocks, width, rank)
        print(f"added adapters on {count} blocks", flush=True)
    elif method == "partial":
        count = 0
        for group in groups:
            chosen = list(group)[-max(1, unfreeze):]
            for block in chosen:
                for parameter in block.parameters():
                    parameter.requires_grad_(True)
                    count += parameter.numel()
        print(f"unfroze last blocks, {count} parameters", flush=True)
    else:
        raise SystemExit(f"unknown method {method}")
    model.embed.weight.requires_grad_(False)


def trainable_state(model):
    named = dict(model.named_parameters())
    return {name: value.detach().cpu() for name, value in model.state_dict().items()
            if name in named and named[name].requires_grad}


def pad_batch(rows, key, device, pad=PAD_ID):
    width = max(len(row[key]) for row in rows)
    tensor = torch.full((len(rows), width), pad, dtype=torch.long, device=device)
    mask = torch.zeros(len(rows), width, dtype=torch.bool, device=device)
    for index, row in enumerate(rows):
        ids = row[key]
        tensor[index, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)
        mask[index, :len(ids)] = True
    return tensor, mask


def causal_loss(model, rows, device):
    tokens, mask = pad_batch(rows, "seq", device)
    labels = torch.full_like(tokens, -100)
    for index, row in enumerate(rows):
        labels[index, :len(row["labels"])] = torch.tensor(row["labels"], dtype=torch.long, device=device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        logits, _ = model(tokens, pad_mask=mask)
    return F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1), ignore_index=-100)


def encdec_loss(model, rows, device):
    src, src_mask = pad_batch(rows, "enc", device)
    tgt, tgt_mask = pad_batch(rows, "dec", device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        memory = model.encode(src, src_mask)
        logits = model.decode(tgt[:, :-1], memory, pad_mask=tgt_mask[:, :-1], memory_mask=src_mask)
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD_ID)


def kd_loss(student_logits, teacher_logits, keep, temp):
    student = F.log_softmax(student_logits.float() / temp, dim=-1)
    teacher = F.softmax(teacher_logits.float() / temp, dim=-1)
    token = F.kl_div(student, teacher, reduction="none").sum(-1)
    return (token * keep).sum() / keep.sum().clamp(min=1) * (temp * temp)


def teacher_causal_summary_logits(teacher, seqs, starts, widths, device):
    rows = [{"seq": seq} for seq in seqs]
    tokens, mask = pad_batch(rows, "seq", device)
    with torch.no_grad():
        logits, _ = teacher(tokens, pad_mask=mask)
    pieces = []
    for index, (start, width) in enumerate(zip(starts, widths)):
        pieces.append(logits[index, start:start + width])
    return pieces


def block_ngrams(logits, produced, n=3):
    for index, seq in enumerate(produced):
        if len(seq) < n - 1:
            continue
        prefix = tuple(seq[-(n - 1):])
        banned = {seq[j + n - 1] for j in range(len(seq) - n + 1) if tuple(seq[j:j + n - 1]) == prefix}
        recent = set(seq[-24:])
        for token in banned | recent:
            logits[index, token] = logits[index, token] - (4.0 if token in banned else 0.8)


def collapse_repeats(text):
    words = text.split()
    for span in (12, 8, 5, 3):
        out, index, n = [], 0, len(words)
        while index < n:
            if index + 2 * span <= n and words[index:index + span] == words[index + span:index + 2 * span]:
                out.extend(words[index:index + span])
                index += span
                while index + span <= n and words[index:index + span] == out[-span:]:
                    index += span
                continue
            out.append(words[index])
            index += 1
        words = out
    return " ".join(words)


@torch.no_grad()
def generate_encdec_bos(model, idx, max_new_tokens, temperature, top_p, eos_id=EOS_ID, pad_id=PAD_ID, min_new_tokens=0):
    pad_mask = idx != pad_id
    memory = model.encode(idx, pad_mask)
    dec = idx.new_full((idx.size(0), 1), BOS_ID)
    produced = [[] for _ in range(idx.size(0))]
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    for _ in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if dec.size(1) >= model.block_size:
            dec = dec[:, -model.block_size:]
        logits = model.decode(dec, memory, memory_mask=pad_mask)
        next_logits = logits[:, -1].float()
        block_ngrams(next_logits, produced)
        if min_new_tokens and eos_id is not None:
            for position in range(idx.size(0)):
                if len(produced[position]) < min_new_tokens:
                    next_logits[position, eos_id] = -1e9
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        next_id = torch.where(active, next_id, torch.full_like(next_id, eos_id or pad_id))
        dec = torch.cat([dec, next_id.unsqueeze(1)], dim=1)
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
    return produced


def chunk_ids(ids, width, overlap=64, cap=6):
    if len(ids) <= width:
        return [list(ids)]
    step = max(1, width - overlap)
    parts = [list(ids[start:start + width]) for start in range(0, len(ids), step)]
    if len(parts) <= cap:
        return parts
    keep = [parts[0]]
    middle = parts[1:-1]
    stride = max(1, math.ceil(len(middle) / max(1, cap - 2)))
    keep.extend(middle[::stride][: cap - 2])
    keep.append(parts[-1])
    return keep


def split_sents(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    parts = re.split(r"(?<=[.?;:])\s+(?=[A-Z(\[])", text)
    sents = [part.strip() for part in parts if len(part.split()) >= 6]
    return sents or ([text[:1500]] if text else ["summary"])


def embed_texts(model, tokenizer, texts, device, cap=128):
    vecs = []
    with torch.no_grad():
        for start in range(0, len(texts), 16):
            batch = texts[start:start + 16]
            ids = [tokenizer.encode(text)[:cap] or [BOS_ID] for text in batch]
            width = max(len(row) for row in ids)
            tensor = torch.full((len(ids), width), PAD_ID, dtype=torch.long, device=device)
            mask = torch.zeros(len(ids), width, dtype=torch.bool, device=device)
            for index, row in enumerate(ids):
                tensor[index, :len(row)] = torch.tensor(row, dtype=torch.long, device=device)
                mask[index, :len(row)] = True
            hidden = hidden_states(model, tensor, mask)
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1).clamp(min=1).unsqueeze(-1)
            vecs.append(F.normalize(pooled.float(), dim=-1))
    return torch.cat(vecs, 0)


def fill_extract(document, draft, model, tokenizer, device, budget):
    sents = split_sents(document)
    if not sents:
        return " ".join(str(document).split()[:budget]) or "summary"
    n = len(sents)
    vecs = embed_texts(model, tokenizer, sents, device)
    doc_vec = F.normalize(vecs.mean(0, keepdim=True), dim=-1).squeeze(0)
    scores = (vecs @ doc_vec).tolist()
    keys = ("held", "appeal", "section", "article", "accordingly", "accused",
            "assessee", "appellant", "respondent", "dismissed", "allowed")
    for index, sent in enumerate(sents):
        frac = index / max(1, n - 1)
        pos = 0.45 if index < 6 else (0.55 if frac >= 0.75 else 0.05)
        if index < 4:
            pos += 0.2
        lex = rouge_n(sent, draft) if draft else 0.0
        low = sent.lower()
        key = 0.08 * sum(word in low for word in keys)
        scores[index] = 0.4 * scores[index] + pos + 0.35 * lex + key
    ranked = sorted(range(n), key=lambda index: scores[index], reverse=True)
    forced = list(range(min(4, n))) + list(range(max(0, n - 8), n))
    chosen, picked, words = set(), [], 0

    def take(index):
        nonlocal words
        if index < 0 or index >= n or index in chosen:
            return False
        piece = sents[index].split()
        if words and words + len(piece) > budget:
            return False
        chosen.add(index)
        picked.append(index)
        words += len(piece)
        return True

    for index in forced + ranked:
        if words >= budget:
            break
        if take(index):
            take(index - 1)
            take(index + 1)
    picked.sort()
    filled = list(picked)
    for left, right in zip(picked, picked[1:]):
        if right == left + 2:
            take(left + 1)
    filled = sorted(chosen)
    text = " ".join(sents[index] for index in filled).strip()
    return text or " ".join(str(document).split()[:budget])


def decode_new(tokenizer, produced, fallback):
    texts = []
    for ids, source in zip(produced, fallback):
        if EOS_ID in ids:
            ids = ids[:ids.index(EOS_ID)]
        text = collapse_repeats(tokenizer.decode(ids).strip())
        if not text:
            text = tokenizer.decode(source[:48]).strip() or "summary"
        texts.append(text)
    return texts


def generate_rows(model, tokenizer, rows, device, src_budget, max_new, temperature=0.0, gen_batch=4, hier=False, min_new=0, extract_words=0):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(rows), gen_batch):
            chunk = rows[start:start + gen_batch]
            prompts = []
            sources = []
            for row in chunk:
                src = list(row["src"])
                if hier and len(src) > src_budget * 2:
                    pieces = []
                    for part in chunk_ids(src, src_budget):
                        mini = [{"src": part, "document": tokenizer.decode(part)}]
                        pieces.append(generate_rows(model, tokenizer, mini, device, src_budget, max(32, max_new // 3),
                                                    temperature, gen_batch=1, hier=False, min_new=0)[0])
                    src = tokenizer.encode(" ".join(pieces))[:src_budget]
                else:
                    src = clip_ids(src, src_budget)
                sources.append(src)
                prompts.append(src if isinstance(model, EncoderDecoderLM) else src + [BOS_ID])
            if max_new <= 0:
                drafts = [""] * len(chunk)
            else:
                width = max(len(prompt) for prompt in prompts)
                tensor = torch.full((len(prompts), width), PAD_ID, dtype=torch.long, device=device)
                for index, prompt in enumerate(prompts):
                    tensor[index, :len(prompt)] = torch.tensor(prompt, dtype=torch.long, device=device)
                if isinstance(model, EncoderDecoderLM):
                    produced = generate_encdec_bos(model, tensor, max_new, temperature, 0.9, min_new_tokens=min_new)
                else:
                    produced = generate(model, tensor, max_new_tokens=max_new, temperature=temperature, top_p=0.9,
                                        eos_id=EOS_ID, pad_id=PAD_ID, min_new_tokens=min_new)
                drafts = decode_new(tokenizer, produced, sources)
            if extract_words:
                drafts = [fill_extract(row.get("document") or tokenizer.decode(row["src"]), draft, model, tokenizer, device, extract_words)
                          for row, draft in zip(chunk, drafts)]
            out.extend(drafts)
    return out


def score_rows(preds, golds):
    r1 = float(np.mean([rouge_n(pred, gold) for pred, gold in zip(preds, golds)]))
    rl = float(np.mean([rouge_l(pred, gold) for pred, gold in zip(preds, golds)]))
    return {"rouge1": r1, "rougeL": rl, "combo": 0.5 * r1 + 0.5 * rl}


def load_adapted(arch, method, rank, alpha, unfreeze, base_weights, peft_path, device):
    model = build_model(arch, vocab_size=8192)
    model.load_state_dict(torch.load(base_weights, map_location="cpu", weights_only=True))
    adapt(model, arch, method, rank, alpha, unfreeze)
    checkpoint = torch.load(peft_path, map_location="cpu", weights_only=False)
    state = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("teacher load missing", len(missing), "unexpected", len(unexpected), flush=True)
    model.to(device)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", choices=("efficient", "decoder", "hybrid", "encdec"), default="decoder")
    parser.add_argument("--method", choices=("lorma", "lora", "adapter", "partial"), default="lorma")
    parser.add_argument("--weights", default=None)
    parser.add_argument("--bpe", default=DEFAULT_BPE)
    parser.add_argument("--data_dir", default=DEFAULT_DATA)
    parser.add_argument("--out", required=True)
    parser.add_argument("--epochs", type=int, default=4)
    parser.add_argument("--max_docs", type=int, default=0)
    parser.add_argument("--max_steps", type=int, default=0)
    parser.add_argument("--train_minutes", type=int, default=0)
    parser.add_argument("--src_tokens", type=int, default=768)
    parser.add_argument("--tgt_tokens", type=int, default=192)
    parser.add_argument("--unfreeze", type=int, default=4)
    parser.add_argument("--rank", type=int, default=16)
    parser.add_argument("--alpha", type=float, default=32.0)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--batch", type=int, default=2)
    parser.add_argument("--warmup", type=int, default=200)
    parser.add_argument("--eval_every", type=int, default=0)
    parser.add_argument("--eval_docs", type=int, default=64)
    parser.add_argument("--gen_tokens", type=int, default=160)
    parser.add_argument("--min_gen", type=int, default=0)
    parser.add_argument("--gen_batch", type=int, default=4)
    parser.add_argument("--val_frac", type=float, default=0.05)
    parser.add_argument("--hier", action="store_true")
    parser.add_argument("--extract_words", type=int, default=0)
    parser.add_argument("--no_train", action="store_true")
    parser.add_argument("--teacher_weights", default="")
    parser.add_argument("--teacher_base", default="")
    parser.add_argument("--teacher_arch", default="efficient")
    parser.add_argument("--teacher_method", default="lorma")
    parser.add_argument("--teacher_unfreeze", type=int, default=4)
    parser.add_argument("--kd_weight", type=float, default=0.5)
    parser.add_argument("--kd_temp", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--require_cuda", action="store_true")
    parser.add_argument("--decode_only", action="store_true")
    parser.add_argument("--resume", action="store_true")
    args = parser.parse_args()
    args.weights = args.weights or DEFAULT_WEIGHTS[args.arch]
    if args.arch == "encdec":
        args.src_tokens = min(args.src_tokens, BLOCK)
        args.tgt_tokens = min(args.tgt_tokens, BLOCK - 2)
    else:
        args.src_tokens = min(args.src_tokens, BLOCK - args.tgt_tokens - 2)

    set_seed(args.seed)
    os.makedirs(args.out, exist_ok=True)
    if args.require_cuda:
        for _ in range(90):
            if torch.cuda.is_available():
                break
            time.sleep(4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if args.require_cuda and device.type != "cuda":
        raise SystemExit("GPU required")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("cuda", torch.cuda.get_device_name(0), flush=True)
    print("device", device, "arch", args.arch, "method", args.method, flush=True)

    root = os.path.join(args.data_dir, "Phase_6_Summ")
    train_rows = read_rows(os.path.join(root, "Train", "train.json"), True)
    test_rows = read_rows(os.path.join(root, "Test", "test.json"), False)
    print("docs", len(train_rows), len(test_rows), flush=True)

    tokenizer = LegalBPE.load(args.bpe)
    rng = random.Random(args.seed)
    order = list(range(len(train_rows)))
    rng.shuffle(order)
    n_val = max(32, int(len(train_rows) * args.val_frac)) if not args.smoke else min(8, len(train_rows))
    n_val = min(n_val, 80)
    val_rows = [train_rows[i] for i in order[:n_val]]
    fit_rows = [train_rows[i] for i in order[n_val:]]
    if args.smoke:
        fit_rows, val_rows, test_rows = fit_rows[:32], val_rows[:4], test_rows[:8]
        args.max_steps = args.max_steps or 20
        args.gen_tokens = min(args.gen_tokens, 32)
        args.eval_docs = min(args.eval_docs, 4)
        args.batch = min(args.batch, 1)
        args.src_tokens = min(args.src_tokens, 256)
        args.tgt_tokens = min(args.tgt_tokens, 64)
    if args.max_docs:
        fit_rows = fit_rows[:args.max_docs]
    if args.decode_only or args.no_train:
        fit_rows = []
        val_rows = val_rows[: max(args.eval_docs, 8)]

    def encode_row(row):
        if "src" in row:
            return
        row["src"] = tokenizer.encode(row["document"])
        row["tgt"] = tokenizer.encode(row["summary"]) if row["summary"] else []

    todo = fit_rows + val_rows + test_rows
    print("encoding", len(todo), "docs", flush=True)
    for index, row in enumerate(todo, 1):
        encode_row(row)
        if index % 500 == 0 or index == len(todo):
            print(f"  encoded {index}/{len(todo)}", flush=True)
    for row in fit_rows:
        if args.arch == "encdec":
            row["enc"] = clip_ids(row["src"], args.src_tokens)
            row["dec"] = [BOS_ID] + row["tgt"][:args.tgt_tokens] + [EOS_ID]
        else:
            seq, labels = pack_causal(row["src"], row["tgt"], args.src_tokens, args.tgt_tokens)
            row["seq"], row["labels"] = seq, labels
    print("train", len(fit_rows), "val", len(val_rows), "src", args.src_tokens, "tgt", args.tgt_tokens, flush=True)

    model = build_model(args.arch, vocab_size=8192)
    model.load_state_dict(torch.load(args.weights, map_location="cpu", weights_only=True))
    adapt(model, args.arch, args.method, args.rank, args.alpha, args.unfreeze)
    model.to(device)
    body = [parameter for parameter in model.parameters() if parameter.requires_grad]
    print(f"trainable {sum(parameter.numel() for parameter in body) / 1e6:.3f}M", flush=True)
    if args.decode_only or args.no_train:
        ckpt = os.path.join(args.out, "best.pt")
        if args.decode_only and os.path.isfile(ckpt):
            checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False)
            model.load_state_dict(checkpoint["model"], strict=False)
            model.to(device)
            print("loaded", ckpt, flush=True)
        val_used = val_rows[: max(args.eval_docs, 8)]
        val_preds = generate_rows(model, tokenizer, val_used, device, args.src_tokens, args.gen_tokens,
                                  gen_batch=args.gen_batch, hier=args.hier, min_new=args.min_gen,
                                  extract_words=args.extract_words)
        report = score_rows(val_preds, [row["summary"] for row in val_used])
        print("val report", json.dumps(report), flush=True)
        with open(os.path.join(args.out, "metrics.json"), "w", encoding="utf-8") as handle:
            json.dump(report, handle, indent=2)
        print("generating test", len(test_rows), flush=True)
        test_preds = generate_rows(model, tokenizer, test_rows, device, args.src_tokens, args.gen_tokens,
                                   gen_batch=args.gen_batch, hier=args.hier, min_new=args.min_gen,
                                   extract_words=args.extract_words)
        predictions = [{"id": row["id"], "summary": pred} for row, pred in zip(test_rows, test_preds)]
        pred_path = os.path.join(args.out, "prediction.json")
        with open(pred_path, "w", encoding="utf-8") as handle:
            json.dump(predictions, handle)
        zip_path = os.path.join(args.out, "submission.zip")
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
            bundle.write(pred_path, arcname="prediction.json")
        print("zip", zip_path, "rows", len(predictions), flush=True)
        return
    resume_path = os.path.join(args.out, "best.pt")
    if args.resume and os.path.isfile(resume_path):
        checkpoint = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"], strict=False)
        model.to(device)
        print("resumed", resume_path, checkpoint.get("score"), flush=True)
    optimizer = torch.optim.AdamW(body, lr=args.lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    teacher = None
    if args.teacher_weights:
        teacher_base = args.teacher_base
        if not teacher_base:
            parent = os.path.dirname(args.weights)
            kaggle_name = os.path.join(parent, f"{args.teacher_arch}_model.pt")
            if os.path.basename(args.weights) == f"{args.arch}_model.pt" and os.path.isfile(kaggle_name):
                teacher_base = kaggle_name
            else:
                teacher_base = DEFAULT_WEIGHTS.get(args.teacher_arch, "")
        if not teacher_base or not os.path.isfile(teacher_base):
            raise SystemExit(f"missing teacher base weights for {args.teacher_arch}: {teacher_base!r}")
        teacher = load_adapted(
            args.teacher_arch, args.teacher_method, args.rank, args.alpha, args.teacher_unfreeze,
            teacher_base, args.teacher_weights, device,
        )
        print("teacher", args.teacher_arch, args.teacher_method, teacher_base, flush=True)

    steps_per_epoch = math.ceil(len(fit_rows) / max(1, args.batch))
    total_steps = steps_per_epoch * args.epochs
    if args.max_steps:
        total_steps = min(total_steps, args.max_steps)
    best_path = os.path.join(args.out, "best.pt")
    best_score = -1.0
    deadline = time.time() + args.train_minutes * 60 if args.train_minutes else 0

    def evaluate(tag):
        nonlocal best_score
        sample = val_rows[:args.eval_docs]
        preds = generate_rows(model, tokenizer, sample, device, args.src_tokens, args.gen_tokens,
                              gen_batch=args.gen_batch, hier=args.hier, min_new=args.min_gen,
                              extract_words=args.extract_words)
        golds = [row["summary"] for row in sample]
        report = score_rows(preds, golds)
        print(f"  val[{len(sample)}] {tag} rouge1 {report['rouge1']:.4f} rougeL {report['rougeL']:.4f} {preds[0][:90]!r}", flush=True)
        if report["combo"] >= best_score:
            best_score = report["combo"]
            torch.save({"args": {k: v for k, v in vars(args).items() if k != "teacher_weights" or v},
                        "score": report, "model": trainable_state(model)}, best_path)
            print("  saved", best_path, flush=True)
        model.train()
        return report

    global_step = 0
    started = time.time()
    loss_fn = encdec_loss if args.arch == "encdec" else causal_loss
    stop = False
    for epoch in range(1, args.epochs + 1):
        model.train()
        for module in model.modules():
            if isinstance(module, nn.Dropout):
                module.eval()
        rng.shuffle(fit_rows)
        recent = 0.0
        seen = 0
        for start in range(0, len(fit_rows), args.batch):
            if args.max_steps and global_step >= args.max_steps:
                stop = True
                break
            if deadline and time.time() >= deadline:
                print("train_minutes reached", args.train_minutes, flush=True)
                stop = True
                break
            set_lrs(optimizer, global_step, total_steps, args.warmup, [args.lr])
            batch = fit_rows[start:start + args.batch]
            loss = loss_fn(model, batch, device)
            if teacher is not None:
                seqs, starts, widths, student_bits, keep_bits = [], [], [], [], []
                if args.arch == "encdec":
                    src, src_mask = pad_batch(batch, "enc", device)
                    tgt, tgt_mask = pad_batch(batch, "dec", device)
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                        memory = model.encode(src, src_mask)
                        student_logits = model.decode(tgt[:, :-1], memory, pad_mask=tgt_mask[:, :-1], memory_mask=src_mask)
                    for index, row in enumerate(batch):
                        packed = clip_ids(row["src"], args.src_tokens) + row["dec"]
                        bos_at = len(packed) - len(row["dec"])
                        if len(packed) > BLOCK:
                            packed = packed[-BLOCK:]
                            bos_at = len(packed) - len(row["dec"])
                        seqs.append(packed)
                        starts.append(max(0, bos_at))
                        widths.append(len(row["dec"]) - 1)
                        student_bits.append(student_logits[index, :len(row["dec"]) - 1])
                        keep_bits.append(torch.ones(len(row["dec"]) - 1, dtype=torch.bool, device=device))
                else:
                    tokens, mask = pad_batch(batch, "seq", device)
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                        student_logits, _ = model(tokens, pad_mask=mask)
                    for index, row in enumerate(batch):
                        first = next((i for i, label in enumerate(row["labels"]) if label != -100), 1)
                        bos_at = max(0, first - 1)
                        width = sum(1 for label in row["labels"] if label != -100)
                        seqs.append(row["seq"])
                        starts.append(bos_at)
                        widths.append(width)
                        student_bits.append(student_logits[index, bos_at:bos_at + width])
                        keep_bits.append(torch.ones(width, dtype=torch.bool, device=device))
                teacher_bits = teacher_causal_summary_logits(teacher, seqs, starts, widths, device)
                kd = 0.0
                count = 0
                for student_piece, teacher_piece, keep in zip(student_bits, teacher_bits, keep_bits):
                    width = min(student_piece.size(0), teacher_piece.size(0), keep.size(0))
                    if width == 0:
                        continue
                    kd = kd + kd_loss(student_piece[:width], teacher_piece[:width], keep[:width], args.kd_temp)
                    count += 1
                if count:
                    loss = (1 - args.kd_weight) * loss + args.kd_weight * (kd / count)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(body, 1.0)
            scaler.step(optimizer)
            scaler.update()
            global_step += 1
            recent += loss.item()
            seen += 1
            if global_step % 50 == 0:
                print(f"  epoch {epoch} step {global_step}/{total_steps} loss {recent / max(1, seen):.4f} {time.time() - started:.0f}s", flush=True)
                recent = 0.0
                seen = 0
            if args.eval_every and global_step % args.eval_every == 0:
                evaluate(f"step {global_step}")
        evaluate(f"epoch {epoch}")
        if stop:
            break

    if os.path.isfile(best_path):
        checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"], strict=False)
        model.to(device)
    val_used = val_rows[: max(args.eval_docs, 24)]
    val_preds = generate_rows(model, tokenizer, val_used, device, args.src_tokens, args.gen_tokens,
                              gen_batch=args.gen_batch, hier=args.hier, min_new=args.min_gen,
                              extract_words=args.extract_words)
    report = score_rows(val_preds, [row["summary"] for row in val_used])
    report["params_trainable_m"] = round(sum(parameter.numel() for parameter in body) / 1e6, 3)
    report["minutes"] = round((time.time() - started) / 60, 1)
    report["steps"] = global_step
    print("val report", json.dumps(report), flush=True)
    with open(os.path.join(args.out, "metrics.json"), "w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2)
    with open(os.path.join(args.out, "val_prediction.json"), "w", encoding="utf-8") as handle:
        json.dump([{"id": row["id"], "summary": pred} for row, pred in zip(val_used, val_preds)], handle)

    print("generating test", len(test_rows), flush=True)
    test_preds = generate_rows(model, tokenizer, test_rows, device, args.src_tokens, args.gen_tokens,
                               gen_batch=args.gen_batch, hier=args.hier, min_new=args.min_gen,
                               extract_words=args.extract_words)
    predictions = [{"id": row["id"], "summary": pred} for row, pred in zip(test_rows, test_preds)]
    pred_path = os.path.join(args.out, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle)
    zip_path = os.path.join(args.out, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    names = zipfile.ZipFile(zip_path).namelist()
    if names != ["prediction.json"]:
        raise SystemExit(f"zip layout {names}")
    print("zip", zip_path, "rows", len(predictions), flush=True)


In [ ]:
_register('phase6_summ')


## 4. Serving path


In [ ]:
print("encdec extractive, word-budget 720")
print("prediction.json: {id: {text: summary}}")
